# Reproducing the external-set results without an LLM

This notebook reruns, from public data only, the parts of the paper's external benchmark that need no language model.
The data are the 500 tasks of `princeton-nlp/SWE-bench_Verified` and the public GitHub repositories behind them.

What it reproduces (153 tasks on 8 repositories, same code as the project repository):
- the task selection (seed 20261024) and the gold-symbol table by kind (function, class, module-level, new file);
- the localization baselines that use no model: BM25 over files, BM25 over units, and personalized PageRank over our call graph;
- the BM25 candidate ceilings of the identification keys (share of tasks whose gold unit is among the 20 cards of the flat key, gold file among the 10 files of the dichotomous key);
- statistics of the repaired graph builder (size, edge types, share of gold symbols that are graph nodes).

Every table is compared with the stored numbers from the project (`results/s3_external`, `results/s3_graphkey/external`), which are embedded as reference files. A summary of PASS/FAIL checks is printed at the end.

What it does not reproduce:
- the identification-key runs (flat and dichotomous), which need Gemma 4 31B on a local GPU or Apple Silicon (`python -m experiments.s3_external key`, paper Section 5);
- every train-set result (Sections 3, 4.3 and 5 on the 129 competition tasks), which needs the competition data and so cannot run in a notebook that declares no competition source;
- the Qwen3-Embedding and RRF rows, unless the flag below is switched on.

The notebook needs internet (Hugging Face and github.com) and no GPU. Expect tens of minutes, mostly for the partial clones and `git archive` of 153 commits.

In [ ]:
import os
# Flags. REPOS = None runs all 8 repositories (the full 153 tasks); a list such as ["pytest-dev/pytest"] runs a subset.
REPOS = None
RUN_GRAPH = True        # build the repaired graph for each task commit and print its statistics (a few minutes)
RUN_EMBEDDING = False   # Qwen3-Embedding-0.6B + RRF rows; off by default, see the markdown cell above
WORKERS = 4             # parallel git archive jobs

In [ ]:
import base64, json, os, subprocess, sys, zlib
from pathlib import Path

WORK = Path(os.environ.get("G4A_REPRO_WORK", "/kaggle/working/g4a-repro"))
if os.environ.get("G4A_REPRO_REPOS"):
    REPOS = os.environ["G4A_REPRO_REPOS"].split(",")  # local dry runs only
if os.environ.get("G4A_REPRO_FAST"):
    RUN_GRAPH = RUN_EMBEDDING = False
FILES = json.loads('{"src/g4a/__init__.py": "eNoVyjEKwCAMRuG9p/jJXDp16lYo9BxBgwpqRGKHnr52fB+PiG4phbHjkkeyNuk4g1SD09LEkiWtB0LXUT2sD4vrLG4RPHwy8OSsjnN6+X83Ilo+KwUeMQ==", "src/g4a/groundtruth.py": "eNrNWVuT08gVftev6NKmComRNQzhIeWsga0sFFRSu9RCKg/GaNtS29aOLCnq9lyA+e/5zunWzZYJD3kIRY2lvpw+l+/cWr7v/6NKZZF/liavSrFtqkOZCdMczE5smmovtlWRiVqadKd07Hmvq0Yome6Ekfo6EmanBiuEUUWhxe0ux3ORl0qLasNr6kbNNvmdaFRd6dxUzb24VY0SKsuNymLvX0rsZY2llVZup6l4Z16WqtlX2ghVpgU2l1uh7/frqhDB5lCmxHYk9gpbs0ikhdQ6xCba67Wn6urQpCoSkkTbqZJoS1FWGchnPYvVTZ6pTKQ0vm1kvYMkWFyV4PMu1wby+77veayXJNkczKFRSSLyfV01BtTLyrAatee1Y9q0j3/oqmyfG9U+Gdls8kJZmpk0kiWA+B2BLE9N1E9FYpOrIrMboPVdka/bxe/waifMfU2KcuNvf/W8N//85e/Jb6/EAofHabWvcWrQ+J9evhSz4GN2EQYv5hE/hC/Ex4uToZcv/dD78Or9h+TdTx/eTFAKPn29DAOjoKcXX+kHDISXX3mYXpPlp8vV449xff+nrwkN2EeeT6ty0w3hHM972Qns8V/xGqe8AlrmnsA/EnwutGn4LddJqW4TUuRcrCtAYyFey0KrdjZThQLOzq34QbwDUvK93FrwifKwX6uGMHgA7rM5JN1XNyq7THey3AIjDNFI1MVBM3hkme7gGbwZeKoBDIBQq4bhEPMpcJOE94FxZZZ5aVbggq0ZZGojD4VJNjIl51hgwaQW3jPyp3Tw74MsSrlXPEIi+X+jHbH1DD8SPnkLfsGm/+O+yg6Feu7z1uu8zPptrVPRFqnvyzQZjjAb9GApWAIkVVnNIbHhd1VaSduxCUE+IHx8oChjZYGuDHSokjzrRaJg0b+tpVYJ4LYHBrpBMij0WcA7lxhb8SCjrcivVTI53ULlZMKGlXaY/G4F1mEaqLrB4RzjAv7LHIRi9tyubcG5stIgSryjHQgyhzKHhTOR5ZuNCG5zhFUYgH6rgxG/8/Bsts3N72KnZAbQUfSi2Gehx5GzRyaFYDrhzaG8tmhLEbCNFhKAM41Mr7FFU+SUiB0tbO3KWw6ucDWjSgP+ZYN9zJE/mwnf+QrUc6MasYdY8hrRbwN2JemMde2YjFsxrbkheau1ThOA9tKqNT008859xVfxCwXUBf90ftFh+cx8UWKEH9TG4JFs6B6fWI2QmbA/rxNoDEE/qHsb4dcaht0Ge+pY10VuAv+jAZavwuWTVcybg7Bb1yiE91LUy6fzlciBgeX86YoyS+DLS/KA9aUfCoUYImrLwqaNAFjEKLGnsN2CsOcgL5Md2W8g0XPxhHDRiYX3fvmm3dGTcONEOraWJEMG/swPx4ucAeIu+MQyywKr0vBkpVP1xUJcTc8Rb7PjWVVMsXIxxQoi7Sg0ztvI6bI9bJcCTkhdY9yLAHguFNIprBV/h4R7eeekFDNxRTY+lbbT9ndKJGDv1sSLBeA//x8p8NvMkMfm5UFxdP4Iv6C1zISkoogLGPJNn+sbWd6bHSmQkQlFo6aBAxN8vG8ih2LAkcVazwRSaf3y2Xzl/RejX1yQloiRbjPFlMqwP4/Jk8jnyPdp3bkJLRwddZmpm8vyUBR+eLwL6d7FiO/eBQRRSeCiFAX53WIUTjpmyR0tY6zhlrcwOjHpiSicdhb2OTquSxb2fcwWh9ZY1jXsHIDH8Jx9umjZ2SbYi/lCuLIv3nPuIjWH1jwk8FnLdCEXqSjYx9QV1MGxCw3C8XDZUxyA4/vX4TFWZ1feJPLHdJ6N6Tw7S8cFalaVy9eJS+VJXga2+OdkEPU1U5+7bUnlMrdpFOUexJmYc77bbQVHxp6P9tgc16WfmxytTUBtxZwp/PT+QyQsfPozx8qmlJHucjRQyBm0B+1Qk/BIQoQ0kwtPAj9OchVTwIsjBEjsfu0qtZ/VJrIsUAF3MsqVId7CiQhNdSSVpf4Xy/lD/IVPiKm4fPA5E/KENUE/d0KJykpQCib9whWS07KMeJzczmcfl6fniR2r4VtEO3Ina053cWSBjHvgbGlVYevelbgQy/aFrcwW3io0iKZpWfMzlVaNRMWfEKxQVCxX4WoiLR9MGwMs8jg+RWyriPUcWVYiZ4++ABctwBwvE2nQwtaxRCRDC2k7Th4BTl24dL4GhpyndU1563Jn/C0SR63PGQdETfm26/hbU1zanoESocxLTm7S3TD8VdhGZFagbC3c1QFfJVTiUdvmPIrbUtU09z3kwbGG8U6DheXbSqzuUlUbNF44++5V01SDUtJpYzkySt9c9V1SJJ7gf7hyZeKB+i3qL5asmT6afHnoC8mSAKPRvquM47YeVZBQEUUfzWs1LyVp4AG6Rd2PCyKCv3qAh9W4rASZcQhYo3ciQKN84lnAS90vCrlfZ1KQ+VoceUOvmaTxvVoZBEMoJgZAXDccEKG4bWwjpjuCIcEnsLtuZHGgKjt0uLRHJClSXY7eE1MToZ86vw52P1cGqnYbBZ2oXfPDjY8Vo6jQ/hMy8wZ/A74l0iJrqhqm2isXGulqQMXbWGRVqhPdpGEHwJq7rgWTW85nf161nQg1ExRbqWCBuVy5Etf3XZPBpUy32HM2tAQpo/PTcna14uI0AZxzkySDIrU/mxbOsXKozKUf+/EfFZzAzudz+CgpICd0NXTvERTKzYZhdzYtbPtk1gcnLjTyARA5dyiIhBt0EYA037ZmruWbtxJhW2xzx6KDyonPdfUOcYhFxOOpzYkUu3K/n2ZtfsNyJDda0kLswR96B68E2Y7zEcgdH7TIO+bL896Xsta7yrzneILz6GoOgr79dbm+B2etwhq00wnd2CXa7QjaB6huRMQGpeGlBSuwDyMjNP8GwtB0oVICNV932PtceLEjK2Kz/RyJbX6jqPbADNeSDHniS64Be+KWKCu57zB8K0siinC18ePLLzWqArJC7frenX6ApLAyu5we1E5jXvt4VyG1cTzrawV3LxrTVKcShI5mvv1sPWWQ6fv5d13c7hL6iBI9Ves/Fv0OoEAt/OYrkfX6LG9vaixf0I3ZjGu2vaILQr5n3pwUZ3aSiyJa4dRFaHIzuSY+gonqa0O1vNnE6o4ucwwvs5tO8zZO2uzmk5UMtL0ccBHbeyDXycBmfshXj7uY7ByEMdUhmQr8g9nM/oK4rCjPaahF1YVMlT95OAUDHBSSn9rAQNaeZmiNg66nK4j1gYpd/uIQ0EcFC5NInPcDG2VsPGeDkx8cXShyN2BjXXdzR9SXPj/7rsqq75N25VKxZdlkdgwyqngqIK9cZZDzJTfv5XU9gY4uaFDPouJBB+huG1kazdfoZ4NA1J3CmO+OZGQ772njE/lTufTzzF8xI1xA0FeHuKhkpoNebWz2xABkSJhLnxsNf/XgnbsF7e/yTgScDyPmSMoxEBzVtoz98vix/bLRFrRWgVQntFeNE5VCiBEnrT/nUIuBvExYMozwjf5DOHl70ld9TQp5nPZjVOTu7HAoCS1Cu3naGf8g3lOeH34VEynUiShpg2x7tStFd9XgLk3RFK41Xb3aLzQ71UfiRXfBOr7bGikUqwBw9X+q1q4OnegImjQSLSeqv6g7ihVlTjnlqIToC4fw+wTXY2lAdCwMnTK4SHhwzY4LS10UCQa1dvdpYmEjyGDEX0WDyqSu3Ap6HE4Nvl24FYOR4ULG0OIknLCrDZYdfeE4s4GAPPxih9JaNoiDDu8Det0nkW9QGoewiFzhUFOZhdS2RnbhLRbpaMa07iCtvSOjLdxv5PLtsHakwxNQGqUCW2YnWd4MAj49WADBhbgLH+p+UCkvr1aIgLeqCUbNQ09UXFI5yFQeki9M5dHAPo9WDzGFUd/7D3gJkws=", "src/g4a/repo_index.py": "eNqNV0tv3DYQvutXEMpFSmUaBXooFtgiRuKiuThG7PayXQhcidplLIkqSdlWHP/3zgz12tUm7R5siZz3fPw4CsPwrqt2umSqzuUz27WqdCxXRmau7FhhdMUEM7LRVjltOmZr0diDdixChUbCn9oxXTB3kKwx+lHlMmd7I5qDjXkQXD9K0CraOnNK1wkTtquz2Xsl3UHnTNQ5y0phLcTBal1fOGkdu+1gs2aFKqVlO5npCv4L1tbKBU/KHZhyljXCHRL2TytKVSjwXYtKJqxUtWS2EeDC6tZkkjn57MgPRmpV1ZSSZMGlY5koS8uDMAyDgJJO06J1rZFpykBUG1SttRMYtQ2CYc264XH/VTXD8xer6+HZCYPxe6u5cILSRK+DiVxlLpm2EshXlrlXwORKtRuEb+HVb7iuUfV+WP/4qQ97/4vge6PbOnemxQL5/fvru/v09ur+j/TzdRAE70ZngS/6n1DRVcDghw5XzDpDb1hVrNG08gBtpzfG3oxtZN98X9PZgjePKtiKWq+gs47eATPp6Ro2Z3JC3ViBonUbWNuyta9JlMtCtKVLC5EhGtcoEWMkfT9RUQ5tjSACwLJ2DlAB2Gk0+HRxQC7eAVYbaVxHb2CXtSqPrCyLmF38hpH4euDPSEACwDB8wX2OJXpdrfzLUKHX8FxdP8PB+YgHyxsjIK8YNhzzShglB+GjSXLroRqNJ4A3HdN12cWkj8Af6oIt2wZj9E6niLopA3SySOElpBBCKDUGTy8JC8kuLG48GKM2ZoU2rMXDSIIksH3tS0e5+YM7+kf0+QiyEkzmPkuK5KQIs3BANKIY1vmmj2yb+CzXlGD09u08FpDyoW630MYA/SqbZnCoFVRepmgiGhFMzndal95v7xO3OUDQIoFEIRQ4jIkW4HQfnRNupTDZgex5ifCS75W7DElUISAuXwgMYR+MkSJPGyItisVGA12u6OjCsfj4abProLXbsUkTEnycAMtaGoB8NJar5xCOW6NJ6JtZ7b9C9ArLoGrrRJ3J2T66jEcjsrTy2BI+6d2X9aRR6VyuQ/MNzZKi/6tbt0Dtmr280iYRcR+0sMwVU5uxcRVWar6IPzwxYKHi+MCNrPSjbIws1DN05DKMj4QhvYorS831jVg2He3Exz4ogAN4cQUHdjFAGSRaxQsx8FAcltp96hs0Thx04NjhKOY53EW5jMLWFRe/hgmTxmhjoXKyKUUm+wR6xIEJwIcnhPQ9MNR7DSyFBBYBQfEbsPSXouu1zwChlKYKkJ6m04G+AfqaYqRjSTx3zJObGSc8olXySFYSwG0OVI5OcfGM1QIsoBBHIg9m5Znhq0jIws2y4lNMXDQ4GkQFV/kcgWctXTln1K51/8ecANn4uAh7RJ7KUso2wuh7in/DPmg6qrm0mUTY1A4WgFbhPoD6QOVgFOiYMBIflGH6qfbsw09q+Ht/r32QxaKUs70zFW3wMqA3b+kK78mZChR84QJDh/sd7Kt9rY0EWrbwVMHttewuwupcWMPGd2MixqJsAWon1DkMTROPTneOt+NMNxl0RuJxRreNMBY4iLR9o+RzJhvH7rraiedrPCiLewBAOwwKFux4bW6bUjlai2Y0NAVyDu3RVICru/uEeVKZ0jiBO/BTdlAlQoN0lJMmpZUUDVkPp9UpHc0wTMIJi06Q0CP7pNnJcWfOEBaOE8g04YuP/JW/kAfuZwx07jc8nU97C0s4qB1dIvNfSGwUns/lKMaz6uQ7PB75fmDstAw/MjqaW8gstcATzLZwjag62vhS+Llyy35im+GFukwd9iLI3UYA26YIpe3SrKTCeeFpVmUDVnqzC7VMY+dOCH5pHaOxrnJTQDudd+cvHzDJPaxRY2kLDsRAjTQtTZ9CCfU/8SVKMCUYF/6uQ/5FQ7HoUG18+S7YzyvY3sZ45g1QIzCJi9AzUW8Mv4VfH1PfYXQXzyguQjYAb+cuQTyn+ImZp/S5+V/j0bnxGWr8/TFrNiYjO7RUbV8VazIaZ32OpIbHvQJymU2YSz4kzXg7z2UMqx9e+zHaz630d5hOM5Ed5GmyqafZWzJOEil8cA8LD7Kb6OqkACSM2ByU2CVSBai8cpy9OcxtQc9RJAODD2AcUlxQ7miZT5M7mSi1yG2EX7N0UiqgG2sjb40KT/2JRlyo/BkiOtdUyjOeAse7Aa4wXj1A5JF/set70wJaKM5UP9DrXOfJQJN6nxTUGBJFm7dVYyOIgQ8fQHEMh5amsyHCPmMQCv4F1lxTIw==", "src/g4a/localize.py": "eNqVWG1z27gR/q5fgaozPdKmGdtXT6eaODfpXTtzM87Vk1y/VKNyIBGSYFEgA4KRnVz+e59dgG+WfE31hRS4WOw++4IHmE6nd+VKFvqzdLo0YilrVWijalF+UlZI8V5V5c8mV48z8bd31zciWutC1YlojHZ1LKTJRaVsXRrSoXJxLzfqvTS7dDJ5WxTC4hWfhZM7JdxWCV3XDd7Uo+O5VrnGGryK0ubKQkGhayfKNS8gdF6LaFpJt53NPjayMHKvpvGktILMEPShTsUvpbi7ewfVQptPZfEJWrbQlU6m0+lksrblXmTZusFKKsuE3lelpdVN6djpejIJYzv1dIAd7d891LfvVnlFq7Io1IqntZp+LBvjlE1ErtayKVyuV65TaZp99SRkLUwVTNn8WaYWqGaaYG2VdDhPJtmH+7uffxW3WDNdlfsKnkZ2Ov/P24t/y4vPlxd/XZwDg+zHt+/+fvdcKvrh9e08SMXRD7dzTFrEv/Ewv7ZjJLSIoefDr/+8h5ZauSi4n+4OFIRY/Ca+TAR+U+TEepqI6aqo6YE40gPxo8e6tH5QOv/ULHTQbsuf4TM9gTZPsjz3IFlo2WDQr7GVn/jL1n9ZSeO1FAU/y6bg1ept+3bYKhOsCU8/RRtX0rOpVXjk7RqmNDzmbMPPtSy8kE9DekMMvaeyZjt8eOitstDMJjgb1uEJiDY9CbLOF+cq9gVPD8bhwFo9FBtA0ywh/HUymWBB4cqdMiifiOpiJrBALC7ecCnM8WcxY7Vlg2/dGGI2X/A4AoA6OyD5hU+dtK4K7VhZ7KfST68FYkCS/Rj9VqVx2jSqG6ykdTWph+hCnAufaWndLKOpgP0YjsMScTeJjKjIBJ49XsFBWZUW5UHZwYxgU6FMhFx7cyuuuSM4thKKODFphP67VNe53tCSY90BmFRWlTI5NPHX0FbwAQhzLLl7+amEeIbq0y7LIkptVG65qgO2HcCLROyuZmJdlJIcuEqvE7Hs/1+mf7nhKP2CpOptIn3p7irxL0sI0p/l+LtbE7yhb0R5zOjl5DQZshgLAyBImyqV1sqnaE6AHU2BC+6pUrdsXTxWID9t8gIq+FvU6kz3SpoojtsYkJpYKBQEXL3sNOTrWdvg2rRr7R4Hf0W2BO/GIcrXaVPl0qlolaLF1Fi0+06udauPzdY5ofQFOU99GOmzia6QjZERF2KNl8sU+L/ChhTePSQuwUdCBQqc2mMxFFkb9XpVWlWHmH9slH0aFBQHEzCbnIHuffg48JknDQz1kfmsbFlH5EcAYOhhaffcXzkvxJmAFxdtdpy3L2d9rF8NojaGGAbsYTf797F1b4y1B63FL90oagN2f1R1JIf9cpy7L3YELuL1KAtXne5EXMZHGfBSQnrQzm/ZgDNSCkBabM7FFUfUUUgJt9jDdd5nwMchtKHK69BFiTFkSKSoOdU9/whq4rbcUIhJ6LUGTSA+IbAlQVWlkKFoP6U4KL3ZEvdQgAFiNpChZZk/MXNJhz2m691NSnSETL6GxYPhlrjQp+/Hn7hFB/OJKmXMrrLl/vom0jk4V0cLEs+cXtwcmApR6LFbqZwmp6zLo7WkFKQOGM271asje7pJ82oRDzs6VC+8IlpiuU9DIfUzybZ41HvnPG2uF6xIkyLOng1ZGF2AQu5AgG6xm8plAVK3GMLA7PKbYHBNVai+aSeDCg7IDHwfZghZ1ZBV5DWv9+0uMldty+Elh0ZgdIsAkLTReQ8K64LdbRaDjBfZxspq+8x1dpfYxhzMg/YXflkEL0F1/2VybcFNkcVMnlnJjBXC2teYza+U7LVnycsnUYPggElTfiaiKppa+O2SxGGu2Cu3LfM6JSrNaD5lJDvzlgDyxGcimwJIBhw4YhbZURSdeLwVKLGytB10oAyaWNA/78smtZ5rTFNwj6t4fnG1WLTbvfbqZf4wOwXNM3sw7uURgMyBgNP+0lA4ZojF/7Lx6//pCUlyf4FkkxL0z2kRckcx0wpOc0v1kZgvjlr2pXjtyZKfFovXt+JmRo2t3ulKoEPhnLFf6k1TIoqkBmcn6ExBOBVTVNuYaXzU7cnOBzIyKD7eD4IBD+IP6NunP4coUHbLPI8e4t+VevBSOh6SU8TXQ9UGfrwSeCX6McBqg8doradffN/9Opt9Oc6Z79LvOGcuF1+nR4AGjdgGiWGe3goHPnnx+KSI/zbyqj3Z5g+hsIfH5KzCMZk6XfRS6qKRISn9kEJ+9m0teRFa2FJUWznkqNc36JjYpWtWhqHvL0/yHJQ3z+XzJDoYTqSVLZdyqQHjUyp+kmZTaLPBTozm4H2rabOkMz1ZCB/QDDQOdDgad92iYjKCr6/4gTPEPmLGOfj3BpnNtBM2rZuiiAxChgkm4MiHh1VZPQXCmavNER8GipwOqEywy7jvrsB4oyITn+IjJJT1QgzToH7NoxuyO9NHPm/BAFhzNgdQL7w7o/KHOWZp6VqC8uAFxrYmj5Bkx8lXb4ma0CLYRV8FsZfrl5Y6XZ1wBTVHtItV9gQKyn3YzxCqc09NeYCZFyFw3nsLmVFq2+Mdu6rs72zYCRKmyij0XTpeX55M571+HGbxzSnGgxz7QKl1f/9e0GUDpyIW8PdUbE9CY4a0MZ1sr6gwgWggy/ltvkvYLOH9/wQJCY54BNpjS9/3PXUge45zhox6Rhbi+azDYtHNBTmrFl50Lx/1HglV8xhx7L4aTjaSMW+IkXm+dQwIYE1lSJpDDYbXrgBrX7O0hoWo7UXtc1E78QcF7DeGfSaMz2iZkEX4TzlU2W+kQUN0WrUvUURPkuliyccoYl5HKCBNn5/onmUNIDEv3qKQRbQFDdX11yJ+7wk3H9PZbBq2luEmVrW3F7zO+NiDkZa3jAuJvrS1pDiO0mW76NihRGzKApsB7RL+/44LacCEuWgSnFXKoq+TaG0l31nSxSqpENyKeNMvq4tdIg5b8Dy68HXIa0k3sIbuauOuMMLtEa///ADmLxamRoJeJOIfdKk2yHu6WQyuzGe7wGu22oUSYnP+RLIjTEjgVScAtTSA9Jv8F6/gt8c=", "src/g4a/external.py": "eNqVV1uP27gVftevINSHkVJZTtKgD8Y6wOyu2+0FSRC73QKGodASbXNHJrUiPZed5r/3OyR1m0kHWD/YMnl4bvzOd47iOF7dW9EqXrO9UOXpzNsbZrm5MSxZ/7yauUX2b9HKgxQVM5e9ETZlUjF7EszwM75OvBGMG7dS6nMjrLRSK68mj6K14o05aWsYbwX7cpQWD+VJ3oovkGn3vK4N0wd3/lXePLCDrAWELdtzIwqoPEubMaMhwW10fMfzVjS6kKoS9/n+IuvKPzOuKkbbx1ZfVGXbiz1BlFfFodXnwgQ/2J1GkBcExtVRVHm0OQk4Jg1TmjWtvpUVQi11Jdix5c2JHXRLzhnByK5ZMK+fOQOs5G0r4e8XqQonv/wLr434EpE30KhIkayQUK+t1Lei5Udn8Cy4ubSUWNHwlltRI3r4yvSlZfpOMRedaNM8iuM4itxeURwuFqeKgslzo1ukUyltOSXdRFFY+8Vo1T23cEWfu3+4RERZCmO8vlLXtSjd6U7hD4gPsPD7Fbe8rLkxot/nppKl9dsNt6da7rutT/gbHH1yFZ3EBrjY0ELGNqv1pvh0vfmp+LzKGLBWayPVsTAP572uTQblLSAAEyWkn19lFK3+s2FLZzSJydG5CHiO0+jH6831ekX7JDVnsbkTDtDFbQB0Tlmq4+jz6tPH9SDnrjmO1h+uP61/+rgZ7XSGsQvd/1gXf19//NA7IO4bKD4LZc3c/KnoXCl8JZAteLX5/C/ES0EPaseADS6tV6sfIfD29ds/v3n99l0U/cGhj83eM3U570XrSsZVqtVUiU0tWPJBK4FTKKk0Z405zFvx60UY1B7QxoFYjtJ1egidWBP3ZX2pqArWq3+uftj87eOHBaPL3RrbZih0y/7LSOsOah8jhk/cPFionFXidu4f44UTybptdxH3qAv+8HSrhsZwkh6n2xw2dfMwD7/TTdOcpLqfVbqc+0dsv33dbZbyRtpZLXir5uM/Y6Ezt02tLcA6Hx7HAtUvYAQ99z9h42sURZU4sFoDfBQYCDBJ6RqGLNHTbuF0tAKlqdhju42lMparUhSyincL1joeaYk7z7xJ6JpzUgqUB6B6rrKATZLmpqkl/FPCJGnaOWEEVSoUGu9CLY1zIRgn93BNU1e9W+qIDc8D+Wf3kxDC/C70LQZdENzu3LpzGGDJmCK3e4Tk0oozXPBW6dNoXeOcQX2LKpHupMx8tORJd4LJA0NqSGkMQ0unPu3VEE3+cem1QVIRQh2kBRi1045Yco/3hAThXJqOcw8l4ztzNTJK1+iywKlrl1EwsC+lvrHdYkm3oF7QEkcLvJMgMNoKeUWTEnVlcqLll1NvGlFiZ7jvZGCO8Y1PYtiSjq3c+USSW6RmgikT73Yhzj0aa0GZTOhrwXCLLl5ipQksPc/N2SF+JEkPsuRqfpVu3+y+5mjNcYe0QHTEvaeEkuNp4dt6B6Z0ukl8ezXy9QrK0erz42+dfqGo8fVE/v8tIMHXflp4eUBgCdo2GowF+4L7vq/13g8cB4Hmgdus+W+ya6+kCa3FSsw96DlK9PeoL5Zw/Cz6UCgHEsjFPYA0wX/IAzbdUp+P/HxTyTaBLWoKS7Q9AUDR8ULfuL9esT03MEu6CWiFuRwO8j6Jc6zHAUZ9187bi0p6y9uYbi1D2mczPM1gbvk4AMJfBT1e7dKvkIvD7EWPsxngBSpchruhJY1v4CeB5TTrrbjy2MajfMc7p4DO0J3gby+MdJc+OL/Whwi4K4yMCeKcoJ3SFpDspjnXCPv2+RQb/fwwFDHNobU+ypLm0KcT4Ehn5uqY7pgPk54by3oAiEqiWy7Hk4dPYuye4533vHkoOsmtcFUqqEr9GmAicoJOLlRlyCSuEjlKPa26+5eOkbdeblDQ6yX6w6wqcmkKJe4Kwrw/bjAcAgfE588GouQbyM16g45+e+uOUx/RWpxSP3EtRhw57QMT7wbgm5bIzXG0i3rkbUfaztv8CEL0wY7o/uDOB5IflLokTVQtGcFpIhA8znnTIMvJI93PCU3bGwEsf73wmuCGtfg7aHK08Z4Ae4PXBVo96+pSu0qgRquo5b/OJkaefmKYKkbCLKbxnjp8mFVY3L0GYMW9CHxNJxpLrZD9USyUXNd4no2/CZKT9fGIXNfetkkXL2bi1Ss/nycm/R0Ohlrsq2ugmBGRL30pTEabzDXxsOM7ezZm5uW3uGMIn7h8+awOHMxGYjRmIvgbUbx0gBA1fqfIjSC664A30tch6yVN09obzoZ8L7u3lEByoXG6xYIghmjxGmUSRz2T0nI8Fl6yehb7K+43HGf+JNs/MFKUDePH+g1eOK04avfOmXSwNm6e93jG4Hsr6hBG92KZ9gRXLjrLqKrwFEYV96LrZiDv8XOM2m0coo53UxBiOOlSFbvC3g4F6Ca9cQ0Gatj6StztaOh7M0ZhGf0PUI5WoA==", "src/g4a/graph_builder.py": "eNrFPWuT20Zy3/krYDiVA2wuVuu7Sl0xRyc6W7ooKUsuaS9xisdgQQDcxS0I8ABwdxl5/3v6MU9gAFKyr6IPWhKY6enp6ff0DH3ff5/vk6LJMy+ts/zitkn2d97mUJRZ3njB/tDk3ssP13Ov7ZIqS5rMK4tNkzRHr67KYxjNZn/Etq2XeNx129Q778djd1dXXlsfmjRvvccCvh46L3/K00NXVLdeUkEL+BB513e59+8f3r31yuSIbbZ1WdaPrdfd5bN9Uz8UGaBGoFsvqPLusW7un7wKcS2L6n7hZYB82kGj3aHsCmo5p/c8rldkl1Wyyy+7/Kmbe3l2K17MuuM+v2QUL7ukuc27y/v8GALWMHjSIQZeCz0BQuulSeVtcm+X7Pd5Ngf8My/J4PmjaHkE/NP7xWzmibG97aFKuwKo8LOXtMcqjY0HaZm0Lfzd1dmhzOeM6D7pAHOYVF7VgGiVxfLz3w5JiXOYe0Ubd3nbwSg8EVi0qkuKCj4JYN7Ftwz+Uo43F8PBi10OCwHoK1TgWQXwgHpZvgWo4l+alGXLH4vdvm66i+QxAU5o8rYuD9QzKGtodUnLzW1aYJK83EY8CBClqQ+3d0TGH96/m2voxj+Bc9J1TbE5dLmX3uFsABLAhBc8iH6Na+YV1TZv8ioFesD02645pF3dtKEeQSDkWVThT0aj6i5vCmqlCLRJ2lx8Cx7v8oqwp4c09QcgOaAHPEkvmnxfG6M2ucCr9XC1Wq9u9MxaWN+2BUInLZHXA4477PIKxg/yCATh5vt8D4veBsCIcbYJbwzIu+S4yWNelUMlUMm8m3rz1wiHCsIb4EQY3LvBrzfAJyCQYiFgZYGPMpC+JO1KFNwUaahmMJu9BY4lLmfhozdK9oDED4AmrDnIWt2RpDFNkWGBD/IkQ5G+aZv05vImq9M2xo9e1tQoLOFsXx5InomPi20BEJibed7bBHTLvoiAXVA3RC9/fPMePsJLIAQ+zOM7kLcyb24i7wcemmXs0NI6zGy0EHbkgVrLE8HYRVUg/rAgW5h9Hw8vqB/ypqyTDPgOpr3Pm+4IrAwwgak8wGKWeDdffnMzh/9/e+O1h+22eIq8d6AAvRtA/obUwQ3okG1RAvFRUg641KRJcokXqpDmUCHp4VNyC0QF/VJlm/opmvm+P5uRMMWgKDpQu3Es2BhaV3WX0AxmM/kMlID42OTyU3ts5UeBDINMYVnzlEkg3gNVEtCXWZF23CZLuoQYP9dt5CNQGEVeZtwQVx1sgGz0I3zlFyCbyAfi+Zt3c+8NUDDZoIKjTyCks9n7d++u4x/fv3r95qdXH7ylF/iSY/y55+OfcPbhP978GH949acfXr29xjYf/ei26PB91NVP9BdY8gE/yL/IEDETuiVARZdf7EElA53pQRzvj2mS3gFd/efZ9asPgMXL63+L37+CAZo8SuvdHggWNH7wPz9fhgGq2fZffsY/MKvw8md6jF/j1f9crr/6S7Q//sPPpI35I70HYdmqRzCTH17+FF+/+ukaxvjtixcvZq++/9Or+Pq/fxRTl9obESTpxg9CedFHoaLws9Yu+M3QCDBM/P2r1wgQuCJ6LXT79/l2jmwSvUTrM3j6HS4sfIPOr//89ruze4ez2ZfeS9MSsO1cwHSA/oAbr0KMohWCjQtobXy2yXGR8TOYNC4osZd62B5A9gZPwW7783B2TcMAnt0BLMNsNgMWBjSSDBYWvY0Y+b0N2irZt3c14IOcCQDevFttjrAm6xAVPDL8CkwGOjTNekEqFmTvJShkWDKPYKCaSDwJCOe3QYXNLAHim5YHUIzh3AN3AaR8cyQ1etHkJQjpA+vFCAUagYM6qcCTAvIqdS5kM8JXCl9c4cXt//qhV2xBfSN9Elhs4z3OR9uEvETlZ0LCT2AQlroHrES+9JufESx15P9BpS56hEAhe6aX5IoIpMFUdduFGnIL9mxHlsN4iP9Iiy69HdmiqMl3oFD3wK/FU+BHl35oNYbp7aKiRWyDkDQn9ULTh2NDj/3RFy+A+rYu+EfUywF1aPdl0QU+gA8XA99ie4eMso2AdxqwejTYLhw0A1y2dwunZwJEWuE4SJrtXYSMFoRRlqOXHPiHbnvxe+DUvGnA8wAa53vw/3Ix1SYHBV4hCMGmhkjE9TZABlkg3YkliZ95KTZ1XWqmDIZ2jTxAodZCWpCE+BYh/jOZ4EthjaRWhe9NXQPfolUSJlkx5z5BL2lJ3VeLi9+uDapSAxzt/pZaQMvVxRWQY4nKFE0qaNKZoCK305TUgLHbAvrJliVwKj0NvW+9K8KVG71YI2tZ5mEM3tVibdIZWMaP/loXCjAMwx1ISABbpABgG84ForAs/6qs24z9vRhcCx4QCU3LI2gkFgt0yVv0Hrwv+S9S/6FoOnAm2N+DdUHhZ5tjkG9B60oPuiYH0KhXhSMjYC7pDzUBfmkd4kkWOBBmO94m6PIel9gsRIy6en9R5g85o0J+PTt1M8MfNsEaXKfntj5jJHLJcYBARi/tcbepS2aYh6SJ0Un/jCkwODELgFOg66DCGTmVhy4GzdgBFYsKrcEL7PqCVAWacPAWIb65QkerArs9974Bp1VQVhggazEF4V38gF4xM4ToQ59lLKaf3BdV5mAX+sYRHOFK33VUp59hYKp7iAjP4BmgiwkefIDOnAI9TrAHmesP1+bztGwd00WSgSdR1i06bDxdEe1A4IAKi+IHjqzF+12+2+RiUvltkh4VJUEf+AixyGRUIWMHkU0wIwg9rIpBOVYy1BPPmsb7DDYy0W29SzXOBTMuipdzub/rnhaa3qgNBkzD1vMRTGPcIyzTBQeIXfL7iUI4LUn0dlMfiOtyerEWdiZ+TMr7uE3BfgebOjsulAu+EszBXpB0x9VTZXPeUmAl4iQCFHlvVYaAl+mIC4BiBnEtpYFw0YvGQ+NIYCFgEMvTkoLHAJq7AhuAE6sNEIhyeg9rWhZtFzQg+Q3ETQF9Q/zDkLXK4x0YcG6sbUIF/ehRtK/3gTbuhJ5XzQwjb/hT1dwjh7fnNCQArIrAxWytxwwfBBQEN6giNcMYUQzHmyaRIsDX3iojW0HikUT3j7F6B6hlKGuovWjpTf9ugLflti/OxxNQqCJeA0DmPnpIygObr3tEqYrAj32sm6xd2+7lxBCICkRYIAewMllM4XhQwWoJPmwPm3gD8nAPDrnWTMR7iNGK/sOnbbfr1oL9yC91vYa1WTHbI9Jk4Ao0+8giGAzVDaKLnyDUT0p6bBDoAfpD6ICpGMBmLhwppHc4wiQPc8KjR2TAL8L8H8z/QXnSkiJ3EQ5LCN4hduaAvshfYOS2WoeDvqnumw76prhwZseBdxnTSqB4Bkivtif1RMOh2NNjJfffG1kS8HXTOzK+KnGLYst52xpkvSEbEbS1l5TwH2gieFRsL7vmeIldLmEmoZJxnBYwIcyLsRuhOU4W49ge0Vma226CM3EAYjYcw2Y9R2jAADnVoglHPRT7suMd5E/7xmZeraUX2oEWTCvMk+BUVlnG/BCYCKllTjDsObiSu7BphBxgyCM8AuD0huR3NqSgHuEtxt5TwCG2num0JfGTcqKVGmZv2mI64SwJk/lH3iZYSL/Vk3FBgIngOUfTffNFdNQUJNWCaWMOvZfcyX4FFtkCg8bZCFlVO1JDVkM0Z66WoO0Ab9e6qSaUYBfAbDvN82AN4gKedk82FuBajLSLhZ+qLTkaNQhxQ7spaW8TpsLcBRhZZ+g72Msw2u9MjGJOg47gNAxrdEdOSwOnZPlndCdKxBiOH+H9a1BA+YxafOldwD/KsntXCxFNsP9hpoAvPvXfTHE3UIYY28HBqIIoHY5aDrw8kB7N1OEwWzLXkbUjNzBImEiTp6SBpqW+caIiYgS0NzHUfZhvLKoDBQDJ5QYzB7STdbm5lKILzxbefZ7vyZPfFo2hejmCPQ7hYlgLU0HVA0qjzY3JrxCrtT2j/CnN950XfDhWXfL0CnMoc+8/UauJz+/z9NC0sF70PRwd0GIPS12o3A3qioBJzhtsTPg5gQitBZRUJvdTQeqNLjMRZg4q6qW4EFiBkJqkus2Dq7mR9XBMRg0VgZwJ37CX01gtinXIis/9Bikx966bQx7akwK4wzmxCWmDPjYFparOYCSW5Ca/BbBgQpmHMSHgHHw4rnNYWtQzhhXbGbEIqIbDVkVvzqTu+y6zmALtpcWkVoLKtIsOdYP0/dyR8h2IF0VUMZkWHkybzkNV/O2AoaYwnjiyThLCX8txoj1JJJQkL9s+a1xhs7Gp7gqz+EZ9Yydl63/ENs9ffiye/SmAhff10rvqOw5Wd2NCScaBgZjPV1/dP9JUjMSKDOPoUUAtHBZ9VYG/grJcOYy4dGsql09TGej0mZWxUsG+Q6njZljTOSVCI9KkZB+VupOttTdRUcYHdQY0jlII3EG6/1L5IURjmphZnQoNSvu/dQorDmY4kIIRjkkJJzktpDCjhDsO29+AD/4RYD3D379Uv6F4E8bhtCghgIiwNrFWuqyIvUvarUS0SdPRPIJwtfg9Jmu38BqiCFRClLgPxM6VR4EY+td+GPa9BsUURbbkJ5rvcXa0a6QrHpb+H3i5vgWgmGhb+vzdZ3W+lOR2lxeoDNzyyqyoWIolmROllvjfaiG36tajoERqbomZucDcQQS9nTTpXSCRCUNirKWc0Fyk7ZaspWGRl3Ilx/GWSTYFRZNyh+5TJvmOxGAlU+zifSTyVuxgxCJ56lC2Q0XLsmOpOVJyRrQkAVDc6tR2uM7QNMiIEnOBBDCGL20V8F9PWWB7lsqMoy6MFw0RlWSU6Tjcrc/JB6Uvadn2srtgd40W4xJuxFCZCELJL5qUfGRRDFf8jzzGc/SR1wl16FagxqLGz1XPUnctnX3Lic4oBJhypRDM70WAWS9JJCTdrj8a6TTY6BWdVbeZkQlqUL/sQGussohlao25pSfxhZjmCZkm6+WizBQTTSSLtFxiwYyENzMFuB3RsZZaMjmcqaTI1MsXoFPLmSsxiyA1EU8ZccrJ2KRKBXeEKsG5xFmAHrcdQKGAtXIlFFdMuAsIT3BQ6Dm3cIEX4WQK7iRYAGkAoB2g5RmKFxhRKhngROTOZ99UwvhBqF/871zVa2tgwtLSwkSAT9LAv1gRnwAs1LRSM6yrM9ItS9Yvc0Mz23QrBeF6Fg+dPavSw4COC2L5LfxOLhp7P6r92uRwfhopRhdm0WYYxA4V9oqRJB+Kh5zgMgFZ7KBMdKXE3mHTsw3ZaaMAnYRZECCZ19RXhwRrL2dqDVmzjvUWatU0O7Q4bHNyTJgIUwIej/iUtd2Y8YBh0Pv7YkmNhtMlL38VEFwCSdDCtdpzNLIWsmBRWJ0DlT7W94f9p+crhnmLONm0I14vcvNDXtJmpKzcMS2oM/bgHVqg7gtnvCHwB+bwtc2geGXpSXExImfpkIiECK2Uo5kuIrCw+Ba0noWEGAj/rBYYduMnmAYWKmIHUrKjuU/q/bUXrHgSa4EczocQoyS8Jmw/Ev2kmKLvyFvum2QIqb7Fd8wQSX/K8MD6zp10/s7177goEZwA3D/UIceCNoof8uYoq/rEBi+a4bmohMJtAXxWV7o4VvoM9mYaF4q1o2lX52YdYvWGBnelgRgigkmIYURJDO1q0b6yDX1kU43hv4aQxTGGYCihwbQksf6KiKvwAz8Op3BEUOQnyZA5+pgYbh+NRGx21mTwLa+IoOtg6J3Cewv6UCLeOeugdlZMSXpo4PuKx4TbF1qY3QVUpvXTJY0Gd84RlKgJNGTK4PSBspra0z65q67R7BXAjZfCGAlyJSb9PfZhLusTGLi/wm5SAswkStpxWsv9RCN/KDvMvSCJjB3P0Alg6AVYLFzvMTcxUMkv1udiAxAAEfrfgcX/h1h+AuUlv/tf+Z9MfjoDIOhP1mUuvoaDZBVt6CoxMCWW5UBVHXEWDQTC2oo8Jzn92fV3RpVnL/18tTAiDqVwKNl7K0o1bfM0nuGV/oPZHEaTaU5JziEXUpLq2EZtl5XFJjZ2VNwZUVkDxVtY3i7p0jsqXW+5nIVSHrQFjC5pdcFwuVDJcv+MjTC3rbVaLM2q+2BYQaJUjN70nDkiFF7hsb2Hz9h/6CO66i1xAUu8VnlWPVwKzKM9FHEsQi46FSz0WVxvz1BfWLq15El6EKJXeWVkNM30LSfq2TwIkYAvixH1jRH6HkvxwLMlgRH1432RAWwYXa5ihOkAUHORCQzIyj+NstJs2t6KESwpEINN+Ge6nJ7VBcsOUUaVxPfFagJWr0ejMFV0rRS+khaChHZ6A8ZqnGM0Ou6FmNBFCAyXZbDMI5jRcm/aAERPHCFOMkU1Gt8orWkyBNaNTjGFseUqQ+gRuoqzDSrQplg5HAMkSwBdsHrLwWxudGLQYhYDj1+WxTrCMrlpRsWxUjk+JE1v46yHqi4/dE9cHOCwmvZnbxWNGNGuPGd08Wv+c7AFb+oJ6ylSw46YrLLzLVUv01JxkuWLkWxiL5xKuyfF/FjgIdzYSGz9cXKF9v5C2x8dCYqGKeoKXIuuf4jHqoVitwYbcN2IHYUp9DhuCDZzRHpQc9CRb9ChecVECjG6LjtgUnWrq/U66idbuN/VGkmGZJ+5fCVhTrBdOJuMHfQZqAqDBruH3qSldYO52W7UrqkFAxjbA7oOUR8COrUOslJ1xXDWvT3UXm0qcYJVn/piQGBKM+OQo2UaY9WHqaPyVK4q0YKscEomGLQ3vZ/2NvnEFaf8hvSytGh7X+xjbMIl6rIIx1keJzkyVV4NLQiDDldXpNslPNbwLxbrxWT5C3NeupYJyiH5TIXab80qalxDcb5M1k+5SIEvrX35T5jxmCTqgi3UF2lI62cXAEqpHJ1ud+6kjNC6LxOu03GSW6mpmpVRYjZqzHSbgdCMAJS1Zy6QU8VqUZJlksZjUb59zE0cjrA1fx/Loe6FFlPaNz1P+wtMba3NJbJigIgzHfI4gzgUYxwEJiUr0WonMhAS2ZdV9bJti9uKtzatFhFbAmftqINZq4r1BfbUKM1p9s7cUjd+xE4NLko8Ort+iBkEiSLkd6SCaGcv5M5cRXXosG+nMNyqreXd8eK6DgiM6uWdY813Jh+OZo92aqVd9L4FRB5wUwRILLYM5sb/LjoPUyjmkmOUVQlqi0jLvS5yaAozRXtwAzCbokuPz8jfaJabHkdOsVJ8KAbCD5rBTsyYoNnl1X1OhyYSsOJ09lbkC9x8Qg7B1RxJ9rgEAT7NiREoWsHZTBfoDFwtEbwRMkj5ySq4YRAnPDmi5c4phSb2DfpmGMfRcI308MijPwPZs3NgSFXU0uCzhRO6OyvaNGmG+lsmWIwSPNKI4N3T4Rl53M9LbnFnFfxVyo4UQL4f3r+j40Z0KhAP/WfDiGhonYTzOOkoGcqPLCh87x+8I3d6QVXfoy6C7/vfiVON5N3QKXK8FUJxuxfwoZHWe7fH70l5+ecK/l7e/CSg8UFfLhMzTYM+eSEzmMyPo9kBa7VtoSLGJltX06NuIFPQQsoUehGL08XCVrxryJAqHjZA8gn2HL75IWlKR6wiqomNYuLRMc+Z6R+L6t3eOU3MXnOL7l1/pmOTqjAVve0YbbQ4rhZNcXvX9WY2gt2Hw6ZNm2Lf30wQ6XB5ckWRMOztwxrlo2YOFQ9205EqyWxYw0f8hh9eMnPlmJfxr0Fk8C+Kju/Q7EXF9x0gCm1ZpC5zQSlMahjl4hycMVl6wdO9xtyeqERa0fO1M9tJ4QGCciunxqGzcwcrjSe5JhNen8BhgTQ8fVsVnojSqfevGqePRxKrq/XJHA65ML9yCsfO4lgusyOJY5Ul4NHVHJq2neuIMZei4y5+wEeXQJUU4kgq2ogHWCY879bV8jostYNta9P9MFUUMYa9fMB+sG9qVvScWIveKPvBGCIwsQYh6vSXbW/G+Doe6VFSn32WAbcRmNFpplFdh2/tjNlUNg3PAyzNQxr4x5VsG61oYmcbERY7fZgS4ul/fJ6bSVZ+QCeXwqH1V3gDNPjsmp/52NptPDetN4prb6JmjRJ46BiWYhYPVbQ6bB46y55oI4ka6rPn4dx4LGhhPVPp2nBAHhdp2OXi2nm8RSxpSvDDGj6h89C/kg1AoFwpTwxDpvypO53OpLrKsTPb42GZOLeNkWj/7DagkOxa2rje1y1u7GET72vcS5cf7h+N56snXbcaJEgneDGnVvCBXPundZ8bIzqJHx32GRjIYI+g+TQYQmEUhqGg8fK8SHtvB9qfFmcjljpHTyjaobZspJlIzqeScXc4aCxYS7Z0lylVOsQdnB9MkypOKD6ka/AC3bY3Q0MeTevQ6y5OzbgrRAyfHP3UuskHqSHrzKHrkCr2YpHQ1KQghV7ICI6kmB79OqUjOki1BCLFiZBDiEO5chWKNTE3Zp+OOZUuGELjOzqW3grzNLrySKUH+rG4gfvaxakTJReDqFYFA6PuGhF4J2sE2nHPbSToXUyWBdNirqrdWoTOo42ni2gUKNoOqHb9GwjOSqQMGeEzU3fVeQrFSkyIhTidnPiMJfwFeYlxxSdWbizDOEHx/yq6OwqdXDJyrFJ67y7nKrqcWREvyNiN1xWZAQ80hOiSIy/amp1cz1EyExxhcWO+DWAyGfQLxGGIsiAvSYhZITvcdj5dJWvvaBun5MbOwMnTOkbhq5tFR2yPVU477yVzNdxQGI+lte0djuH9jRUx4YLkbauvTPyVoieD2lwzTpVP/aKxflbqdOWDZeimqwBsD2O8BiLgxCreaoL3VtKyYV99p9LJYbRHPIazcQ3TZM2G3XaAtSwpYXvd2yiCroIl1oOAZQwvssbqXsMe/Mk6tYR9xRy8GtRJfJGVuiGt3nqH6r4CwrAv3t4lGd6gbVwb2I7hZJ47OBcrZ6GKAUwWqrxwl385inaw804W65i1wEKviQMa5m0s57PzORekDHA0BUlcmDKZnzMud3lMim4KtJiTvsnlXMCOW2MGoekp4ObpcWdKeCyVJY8ezvGWa3m+AveE6MOVKuPj0xbfWkV8vcGVvGBN2ujog5K1fm8sE5soeaf8AuA6dNdO1L45Lt85nRZ0pgTPrYVTIz47ajnPL4sbIxSpu4VrMoZGQ0qdThAShfXJY7tspMm3k9Rz1s3hJb9o1AHouIc3xpDmLEc8l8/GlpFanD2LKa9blZOcGrRv8LqQvfDTi8xXJ3/2/I2yniVdIjL75St3OiOuldt3SVn29NpWXneFqdiJ0HXbj5O3ahOXaDJ0MuiJfCuKyw37NaGS1AXVbkfEmbjfjqXtP1n/Yi/pqH+W7rQ3CRxRvu3l0K6BM3o/T1P0uXmYhDiR6hNRJy4vw2pPx56pO9jVIOans7iIZxieNac05FK+80VgbEOFLq76ZiF+2OPvs6XCV2+15NdkzmMljqpIddDBvLjLzb1F9jR6SdkZZx/oSLRd6MdBdBAuJvjwC+17C45BIe+HhyfKeI0prKpIHn2PGrVJOveuaJ9Un4Toc7ODSkuE53aDzXa9UxPG+eT+pUG/vJg5abuzzoU6ypqsGdOK0R3TdmKzGqtq6gk6dHVe+Mgi8b26S1b+ZgbdTUxDgMLfw6rkGe7X4dFTccWE9WMfkTM5Ay0pN4PnzO2LKUbzNL0SMeg1arKGzIA/UUS/0hAQc1uGgSChaTiRAyRCTYw4sDg0O7SaI+kfJ2rdGQnAHlZijKngpCca6kcreEwFRPggVSRudhg5rtfcqsu2qCvu24TuO23xtbrW9pwcHG31nLs7P5mFI0ifmHcbMchjNfHWb4EQKRuqijcyQWp1WWPoGw26hR0su282QK7vMNlAPsMXbhR7GqO77aYcCOs6DPFjJ3QrAtaJ9Qq/8NGkj4FS7nZsGdrc+JGC4cYHPF9MHjqw8MPmJm0NNu4T10y3wXtwEK373EdoTQca9UVaPuJMiR6s2CS/9g/e76aVtTiKt0oH1eeWyVeho42dLIW1S+bFy3X/+KhxVM950R1R0PyBGkFHeeYv7KdF8WaBi4u/XylJV8d/betKrBb+WEusFoqur1fF7wvr8hyi6Ee/yPwFOa/4O0PQT3/ryA3HbTD6cTkfD8zTd75q0pduBD1Tv+LmEHMfmZ1acZLFV5fM0EP1Dd/Q5Tn8WPw+nK8v1aHn5m/H+WIVGWn+/Dxzb6FRitbggWKtj5IatxtqhmBflYhE5WALD6nAv6kHX1okEeka+IK4g0aGTy+eCSzuEuJN8aECT+AGpUcfffkTf9AXg1P8+SP1W3/6mfz60Vcr7C/0akML8bOG8NS//V0SUYdYPnwergvpvBYph3+RzoigL3z0Z3FbNAGICVjgvPH4HJbzfZ9uU5a/ysY/yWj8pOJH8cMSRNvnyHtN1yXTDxVuk6JEV4iqJ2GAoizxCnH8BTj9a2mqjAnzUfLmZsaX9eMmwvttrdueN5GUHDWBcDjpGPGL5a8OTfwA0zl0kCMbNB3/jafQgBjO/g/da5eJ", "src/g4a/embed_rank.py": "eNp9Vk1z2zYQvfNX7PBENjQl20kOmlGmbeyZeqbph5ueWA0GIlcSKhKkAVCWkvF/7y4/RUeNDjYJYN/uvn27oO/7d6gtgkFnFB5kDs/K7UBCXqb04vDoZmmZIWCxxixTegsFveYRpDLdYQbrE6Sldqgd7KTdxZ73t1bOgttJB9IgqIz2FIPJ1JTW0vGi4BNBUVoH5YaOYhE2Z1snhFrqFGPP933P25iyACE2tasNCgGqqEpD0FqXTjpVaut53RoHkKt1//qvLXVrXknHG73tH/Q6GOm6qE4gLeiqc7Z9K2ODVSmUzvDYGz3SygMvRMAZet6n3+/ufxUPd7AE/89n1DP+c3t13xN1NY/f/+x7H3/6+Mu9uHt4pHPsOPAz6eQsQ6MOmM0oZdFQ6Yee52W4gZrQBRMf1IvGVQhXH8A6s/CAflSp2mjY+F/rmPN6+UfT01Mtcy0LbN/YOlnczOfz1YvfwYo9ngLeWDDWRcyOv9ju5HVzNEbNxQ/CMN7hMVNbtC7gONNcUiWHVB+l3mOH1fgi6igJEVjMN1GrGKGyxjPR0DMX0eGDSrFf94vK+nRcHoXFJ5Gj3rrdAhRpawnvrm+aoH8rNbae+NcUzCILMEXhjNR2U5oCje3r9le3+Xnc8wZzji9uwiMPF04Gfeh9qMv2X3gBIZ7GTYDThdEkr7e820Gz1nKZYuDPKHlfCP8V+gFTwZUmm1FMM1YAI73Eujr5Uwsq9XctaN/G3B7f2tkFZCp1CVUkYuZXhMEn47yUmQ0m8BS5zFqphiGozdR5jEdlnQ1CwJwmzNeXb7KyhK2rBjmYZDqC9SuvwMjqC9I0CYJ5dF6CLToxqGEYWSJTBU05mhUUZ8TGG/Lpbm9Cb5As7dN86QTLKREROblkIlYRrKVLd8KqL9jr8faSHAtlLc/IZWMbMJMxS5SJDRyQqMCRfeuAk2y7MgQaZrw+VCEcNUCneLeDHn2NjTssaXwm12d8dP3b2Z6nsRwfiRESu8zpceTMLj+bGqOJt0s/uyufRWXKrUFrxVqaJam9dxjCB3g3n4extO5UYXDOfA+wllRP4ousBl2Mu8yYilrSkEY1GumGfMIpGQN5Sccqa7eBfwPq/8R3sE6m+yAZliNmcTVGMDRQXOwzZYKKLirtOnqgkaUo983raETAVh5wquoILiQ4bZlnoxy2HdU0XUaX09h0rIpBsDSk9oLvCtuJVmXHxfktRRzVOA57V1c5JmeSphh1Jo2Rp9XIYyvMJSRnt1DYVKHmEpCPuPG5mibQdU9jPeb2qRcjp5wkF+ozbYjViPp0UcdJk1OyeMs3G6VAuisqJ/jiW/pPNZqT/x0xh8n8LGy+h+BHeBpWSkN3cisKabaW7o7gitSwp6+ApU8qWed4Npm7KzMZGEnUKq5V1iqWU2rwKEjbXcDGbIIfuGoc0GS47PuZ8n7elGrYautCn0GPmCpKlr+iGAE2NU8z/niyeKCeaJcZGEp6508qsMQKkJwKG/OHVJN0Who8H/BNN3KbdNOZgzcc/BDnpBOrkghhyGk7mteNyG4SPsfQzRtP5oBXIpjHlOUbuI7ndCUFe3okWF4Iz79FmH+ka4GNiSE8LXNZrDMJJPKr1sGRuvQ/ZWVxJg==", "experiments/__init__.py": "eNoDAAAAAAE=", "experiments/s3_external.py": "eNrNO2uT28aR3/kr5pCqEJBALLmKnDLkVUW2N2edLEWnlXO5QlAokBwuEeLBYIDVMgzz26+7ZwYYPLhe2bqrY7ksYKbR3dPT75m1LOvm2dJnJd+XxbpecVZtObu5ZGmxitPkH3GVFDlb8ny1zeJyx+BlXy/TZIVfFIIVdSWStfxqVWR7XiX0hX3zX9cz+oz9mZfJJuFrx5tMfhLxLWd2fcfKGjAdqi3AzjLG7/cAlfG8Ep54FvH7ipd5nLJvVtn6peNPGPwET/mqYqM/EWf7FLiIxY4la8FmL02UFwbKCGGE9zeBTOacA/A6ruILPX8hPnHiO7pTfBNs6kge8ngvtkUlRni4TSoWl6ttcsdZsWFPvP2BxRXjMchgGQsegXwygAHeehQ10guiUZV1tR1fJrst0vVsH1eA8rYs6nytoAc45SxNSv5dti8TkAXt1C7BL+NlyokicpcmOR9b1bdvL5+zTZJycVHnSSVc9v79B5fxbMnX6yS/ddmHD39E+iUXddqV9cURtoAErngQdQZadKC3E5EueVUm/A62eviriv3scj5nSJeVcQ5c3wq2KUq5Bn5AJRLM3h6WZbJmMaxpmV0+d85x09CKnkh+iAPEE8xmaYJ78y7EZ+R4hmrkeR4N3MVlEqPw9EBWrLl6g9+PP75FNEIrVJbeg7aWoECzTck5+3cOy35BPGUodKVLUhij+0x7w56yfZyUHJZVgHZUZbwfrH7ynzUHHBUskiVyc8GQ4eOMiSquOOo/GG16AN4KtkUNiBDYcVksWJKDqXsTy7Imk01ZZCyKNnVVlzyKWJLtixI0Os+LiryAmEz0WHm7j0vB9TsKUz9XYHIS16pI0WDxS41szTcxbMs6WVUSBlR5myZLPf8eXhsqeZ2hBQmW7xV3t7+LNWTjIWAenpt5T/ktrgFJdSOcjpT+oiZFchjVRQ3Q5HBgvy/hnQPONIqraNcSQv8XgR3xe03qA4y8xgGXrcDo+VpOTyZ/+ukju6LF2daIYlrO5PW776//En3/+gPAwbB3/ZeP7IJZ9L01uX77bfTdq+9+uO7Ogg1GRMhi7DegcLAnsOOMOOz7Y4KbvLkBDPbCZbDIxdxll3Nn8ub6v6Nm/JmaciaTCWwWRIF4HZGt2HufVkDWlSaiCnAXQ19bcV3mLEBID78RduqQrqaoY3sQFuBBvbMdT+zTpCJ3YzshSzYwze8BIbwyngrOgtCkTjKw0SR9hiSJgUbUfmPDV2xjHREsmOLWTENJyJ5eTJ1gEZ4iNWm44ml4skz+zV2zUdDaM0eop8QD2E2zVy7S1ZJCzxzBim3yuZJVF2LRvd8yS6xXNUSqQPAqAIsOXaaflChr9DtX7Fh78EQirFGEgMgjjZRuk9QXwaRH2CMIEQ4smrKkYPWXNCa/JCT4JUhLBFNcGIjB9/Hl73Wc5nHGUS6EVxh4xSFbFqnCLAILYwi8/NsVs8Ab1im3Tuy3xL4pUqLsSqpKUjJ08NJWcV1ZD9oUTUVofmTX5YQAeOmR9iv7aKxB0gFgXqJhlN61DkofaNB2DAgPXXbqbeN0A3oG9rLZL77CdObt+xv0nP+6vGebWIBFgu3EwPj+2eULMCEBigqRiAwKx9rIJ/Czl3Pva/iZS5b0YLG/YbNf+dM5z4WReFzIDZGyhPQokjB2FJN6vStyruRKikRqTBARDCiRYK6g5sjG8B30UM2uIHWohFRhVE1Q47wKAd5w3zYMSWjUk0QqmpCEWxwBIg6SMLDQJK0wZE+v2EJuKtD++OrmzU30Hzd/eud9KpOKSw9BTmQNzl/YxwafJYq6XHHLZxbmMSteFfksT/cXTZ4Z6TyT2RUXlWO58BGEY/gEad1cX3/vGui4ik0wfQTPYlvg4S1U7TvcVpSidEZ30o/tXJzIFaofr7/7+Bq4Bp4zkOnJQCwXbknp2fINnAa4cojHwDfuAszC/08o1zXE56uFA6He+mtuSYlSomZ3v095bsM3jnY3tPFaJ4Z7T0muucM0oPa3msMMxmkP0rNNRDQaa8HF5i6raLEQgjnGFHJ9YMYLp93hbbzW6tX1kk7jzhvYvYLkucDkQn9gVwYILXpjBcf8dHHE5RJN5xSyYxVMDfGBd2LHqfTVU9wx5IT2arqsk7SawvTew+wHg00VCcwGLtiCf+V7i82Jvf2WWZNuurWx7OOIQNgMZOV7881JoDpt0lpsrz6WNTd3gazxM3egLD7hnBkvDa2PIukSo8i20IBWaSwE+HTHiwVpBeLEpSriTSIB8oSfzBHJcyPJsDW3Dz99/CF6/+rjD6a5WZb3tyLJTQZKrY+EqkRUyLGj3IPedgoIEcYBKTL4EKEMHbYwKjIVOdjyQIWHD6KUyo12UUF2mcIIZMP2yoPkvMa8QJHKKQsDejklbPAwl9ihrECPYvooyknQUfU9VRpny3XsswASHvwvbBUdIjSt7h/JXqs4LaHVcoiegM5MRFqVvd247LaG6Sb2lxTxWwjFNjg9yOBT+3ZjTtHSmqnaNAXAqZcYVI3zNACCueFK1dAiHCUEM5fhgI42N6mmZG+08tML9imBgpI2jphHz05Z7lGupgNBazjKtZysVrDaXepVaE/pD+ydgfs9+cpcjnewMPCLLXUYWTQjktodrAdpfZEA25S+rUE3QzbUOaJn1k36b6QqKoEpy43MVzJebQsKvojAU6/SDLMlWk+TAaH7ivODnaG/KiuBkrUxsbekGWcoRIVAZcfIyyN8PDoGyvPKwIw9VuiDyjdmbeT3XQ/hyGwRKhcv262T0obiAlsp5P6g/EcPHxU75Q1NnxZ+wRjzedYn14wG01lw2DPKRIgaXQnAqVI5akplqzUyWA+Kr4PLZ33k4MHIOmlG2ikM5ZFkilJxn4I3mKTbizoNHGXHGq52CIE51iT/cspE22T3asOonaNKXWC/URYtdlOh/A4/oIkZu4JsHmtgTaPPcSefB/y9WhrZcaWAgdUOcdL+tEtErXFARGM3FtKr0U1C4DLG2cSSsFf921KM57iCav8BdrptAZOFL0UfDf88fZj1Wh7+l0QARJ6ihB+SQ7mxH+TFbXcOHc2DW/drZSf4kNEyTsBR3hygmsuu7xMMM3W+y4tPWvXZMdPRyjSOHRrHm5sRhC7b4sLbPpCtDAG90G64BnAeARTY2ekC4SL54R8g0qF76E1uk0rOIAUgNEAGeyOFCYVJXlRGNHoEn6rpdVuP8tnlFWHP8kqTZ3lF7+/F+z3P15hI/OK8/uEsvJeEI34bm2sXYLqdTjPkyr8mxcXWLKxQdmihfDByAoRym5DcYcHsbncZMAgjUFP4XTpGNmYQkUANEZmbnOPGN9pxzUdqULaYZKoMz+yfsrKVgYFSG/zM7wTwsisSVD9E0JTFOKljHboMfJLup6grMyMnjnrJOHXw2jSRH5CMgPSJr+3jrke5sckSmbAuLHzenYwEAeoFyXMAqMIh44pASYcDaDt2IrTK2fIjVKkirhwJsvcSqE5zNecMLQahYLYbuqknZpGFSB6BLCVrLaBqyzmto5Y7XCZYT8EHql0Jq2yJgkCDLAwkGNkcnurYxLANvGYcWEUZOBBun1E2SRIh2hLKgtUMMTYJxjnGmzQDqaLVEhWzzUXbCei0eo7psG6DDpWyl1avCrmPVuMRF9i/ad6ed94Wc3xtHNKi8/a8O9cFvWxflZ8DgLBTEf01P04lv1P/m8XvThY6CuVAwBOu/JeL56o9ukKRIetOq9LjCZZGfszGcKKoYFe8Ww5FsdJHewobN3UcJOc925yhmM2VmwoUSYio3eXkvXrOh9g3J0pTtcVTB+qrbkkHQIhUgrWp2tRlx5OjP5ZDjizF+sVTpJzhsDGi3IxReZzx4L1aKsha+QZmkup2sklXZnGuTKZcI6EJdbG1M4utDLfiQpVbO+07MDEJz7rmh2NCLzqQTyL/2Tq61n92PJbp2cRj4o907cNgCzqMEydmq7oauyulPhjRTrsX+k6O1cNkLrsTjr5A4Y3npZPJn199eP3q3Uc6frLkOa7ez45aNUe3v7QmHyu8NUGUAFXqzTHveKEdVG1rrV90Iz6ML4QHx7DPi6GyXzA2xDRQ+LmF9lpG8OOdPyzuRyt7aV8bqz3/Pt6dtJWd6JO7gQzUOZPsIQCph7F4BWR+thUrOxpBh0cuMj/024PwL9cNWGbnisTxMr9TMhHL4He3yAIseditUmn42GbidgR34TAhXxV5leR1twQud1TXyoMGXQhbKlk4U1edXwEWWr0yZivzzs4pyiN6GPJ2BUyWu8C/nM/DU/dMwsBPGbj9CzP8kQwepd+KXneAW3kCyVVaCG6bDgHcx4OuAOaj/lWAN/zwoxpSJ4vaso/VoEn2gK2fvmSHrb0Ac8akfcq21f58cQOHBI6ObYC6CsPomNW3HXdjlwO1gRGFTb3hGSto5iiTmpJaN2wOYDW3RKlU3nT5KSfRnxkeJobKYOt3rLdK1kZ8lfrno8tZc6ghsyQHLwqJdlGC838h78PIvScasdFeBu8LmABnFlfJHe97gK5/x2IASaPn7/v1RzoEPN0gVgJA1O3iPOTvFDtIeND/BDyy5RksHuuW8u5pgrlYeTGKiiP20hgaYl6WPN4NmipKQwb6N9JbxPQONKgBpVtW450O4M6GhboavUtfOtohj390dvkNBzkJfAU+Lq6oI4rsoJvGAeWmcR7Kmm1RFVlRi/Geiuxg5Q85cLe1/UCtIqTNG2/TPMqzo0Qam/Rb4UjL9Gk97lnZDH5mXODCU2+B/xxDj7VPVjuhpugZxrCm0mP0/DnUQEjZvoqqYsfzBrE5hgQKvOmJJ/dduMH45xAWHNSCzuVlgY0I1RimIBQgPzVC+HQmNj4QIzt3OEcPLUyjHf2sKSEB9AShQm4tRFzc05PPaAeu5Eme3hBw/m0lfHWkE0C859HZzAU47N8CVwBsPSAzoDzYjxODf9nRENdon041COguY9TcZbRjH/st+Touy/gAqUz3Nffx2gmI6vl8PsdrUnytR+bGRSoqllXNrP7Rl9LyWzw53uNK10XmqTZUBOM2YlPZNB7TebGoDntuq07QjC27I/KWLDKOZcA6ABQesMJveSnsubydsXb0v04oGzMyDkZUYMX5LbdzVVPqu1FU568VsKMLf+AYSuEV+Agob22i6rJL7/nPQHz9ewQxr4ecq8D/H2Q9dFt2rAlgZB9K6ofPTD46fCC6k+ow5dRG0CW4/fPJC32sr0RlWZG3nxtyeqAGIwQUSNPUtkE9ocjIKEbBqlTRkcmOTr7WLR0MCv8HueDxTnMQNRycpKjwCiApuoElACyh1X6u5IDzTgjeEA8J7tAZqUZVZ2Vh0/gZXNigHjH2GPESVhC2XRhi4jTpJXhyHx4qCZtU6jEnxY3T7SSTSLphYVAD4mjntL6RkeWMnBzt6DymKgP8LljM/dkiDIe53vhBFknxSpEctoe7ReUhUBkRKllo6MnkUadrZe94iNpgSDnU5zrHJ0+OG2qe0hmQ3zlnKnf6kAkvorSnaXS7+PSzwZhQI08jqDtHbR3sdGXZOZ26d1rUpXlSBV/ebZF64+hDv067uW01/+GZpXS5WWR/Jd3m6nGK+k03rKf+N5fzkWbx14POLYDQl5gdTf2Xvz8dp3sIo+oR4ig8faU51X8kAAbSGgPuCrgsoVt3w36FOhcBBlj3sCAog5XZdRShujDW8Hcy7nCKim50/Jzaq2vbpIEue9JTVydsnaXch1YjZYqIWQRGeKwI7QG/9wa/yBJyDAwRd0R47s0J4p5UQqWgbj+ddNs0zzjWUfKVio4Se/IEZWdksorFQXJKPBuZIzzh7Qx1L6M0L2Q06oJUxvQEScK+gLaMtfqlwhAfoCN4h/EIxPERcq0j0AV9wcfO9VFIT9/3/3jExlQKk1j6aynsVJOeL4yLWwPn3upUDCka3peDPYLdoSzNDu4DS+EI2z2QriNUsTSnbZHxgcool8kXo4iisV6wMXYJthuFGstDOnxc9nwt/a1BvOSpy6DgokM/W9JEC4Bibsa65HA1IIAeTKvUBFEaEMbnPUCUijPi+8EXp4XLtgle5+snv8jl0JMbt+JoNUD3uPafolaw4JgW8tFlx20iH8MXwJhYFeUaa+wjfm3b91CoHxzHw66/A0H/TqiZA0zcN+NW9/gaW2aPPsIGoM4ptkw7szjBtLeTbcZ7eRGO/mLIe1Xe1lj+vsc33fIRNTbnY1CqNaatSwKFzBqq/+rKglzWwlr57zWKsNc4XEmbl9e5ycb1pWh8oThPo5JlM0IDHaInidmrhhVz1NInILMGg8xK6XRmHFRoGJqN1YJtazZTJyjAUI79jSvrKV6FlWXJ1S8701I54wg7hn+WMAN2dCdmnB9Mz2x9PuOcw6H/VM7EIWF3IzyBiunZASZqKgEaLLuuEmxaaE7m5z55gPgXWOwQB/Wjzu2ecm2W6WbU9iC4VHASBOLU9zSPWnV94285Omrsd+/6N1rtt7fPUVsa5fO7l1hV5jVUZH/0vJbuM2rF8btnby75CDUKTxp1F6N6OwXUwoORUDbqwUWAG48ijIFRRE21KEKHEUXqrpn0HpP/AVA+jxQ=", "experiments/s3_external_tasks.json": "eNrFl8+O0zAQxu99impPIFHVnrE9Hu48ARIcEIqybYBASas2oF2t9t1xtnXTbjx2b6iHuPl9Hvvzn7HzNJvfHbZ/9qvm7v38brdvu1XTb7tFt9ktP37+sLhvutWP6lOzb7+1zXr+pm8O/du7d0OtplmHOqDAaQXm+GrTrPp224X3T7N5iPc46Bfr5u/yWBwaqTebIcBA13VfLx/q/b5+fEU2bRfrDcUrWh/6/Xb3uDw9r9hh96PtHhbr7Wp5LL508YhW7a+2X2yaet8tL/+Mkt91v9ts+017vxyLI17/rLvv2+Xx8fJ6Nn8ejK+2f7r+kHOtOWkZQPSrlWD2VOf/OG27Q1+HNVK168Hvl2u/VXUqa6WsjnOZoj5H0TqRWnAgQ4dyXEueZOgVZ6BRInSa5bAubA8REiibgSgPAiEYGRrMtGlYtkIWdAZSBrKX2/TIfLnjq+q4/hfA4wBcAVQsAD0uq2uAUihHlAbMmARGkRUAmzS4WLDXwAELwJMABOeGXboNZ1x6SJwVht0RCDXIpA06Ri8AhiQgEJyHdZ0edsJxPmI2HJbQS9nYcd6n0CmTgS5Tk0mJ0KF3MrTgZchKtkLKy20SjKt1Ar1nf3UCVtWpsNBhK5HEUMUhT7BzmkkwlNtDY+WYllhkTGI/w/4S+2J0TDAJhopl5myGyfWs8jLDTD15zAyz1BfSTrJOKM4QOSWjmK0nyJO4VLwneH2PqqpTWauQ53I0WkhSp02OEoqUbDyyUpCMz0BLMmSdCctetuo1yDU9xEWVgmF5yNBmrHgXc3YCMqDKQOsz0GfCYsYnn4+dFLQc4cXtM+CLf2HWfTxusiqOOzGn0uF2V1aFobBlFWowt6jwplgGblAh3NB7NDE/5lRGxcycV8ULSFYVfjeobuvX+S6UVdF5V+ZU7MsewWoFN6jI0A0qpnKLYTedpnH8pqqqsfwyCJhVgDKFGBCSpcsrtHWFGAjxbigqjIa8woQv/4IiTFRBYeMFTFQ4pILCF2NwSWE1FNyG+16hH9ZQwa2l+GUhKtgVehp2kS4o0BS8uPNRffzAr6rjc/gwjx+zE8LxAH9N9HmWJwRQIj4m4gnheNN/TcA5TJPxgjshRglOkbxAxgQ6IQaNRFioY228Uk3IOSFOiJfGOpxfwug4sIIf55VUhwOZzb/Onmf/AOULRDs=", "g4a_repro.py": "eNq9W3tz28Z2/5+fYovMVMCEAvV0E9byjCPTN66dRJWU9nYYDgYEliQuQQDGQzLD8rv3d84ugAVJWU7iW44tAvs4e96P3aVlWW/y6EHmYpbmolxIMb/wj3OZ5alI0lJO03QpbLmayjCUoYgSHtP0TNeiCPIoK4vBtIri0OOZXt3vZmvH7fVuq6TQ844/fPhJyE+lzBM/Pi5kKYJ0lVWlX0YpBs3ydCWyahpHgQj90hdpEq+Fn4Q8zM8lw1mJx6hcMMSiTHMZ9nI5k7lMAixRAde8EDbGVjHwKs69er2+MBrnuZ8tlnI9qHsdAPRLUSyirIXfUNIbfcpkUCo6HtN8GSVzEUY52tJ8LcpULNI4FEUeDMDBPmjMZB6tZILVmIAGxYGwOzx8zKNS0wVmWZbV6zEbPG9WlVUuPU9EqyzNS4DBJMWpXk+3zX+Psvr5H0Wa1M8rv1zUz0U1hVQCWRRNy7p5LIGkWjBI4xjksCB053VaJeBOX4Ry5oNzYRSU9eAkqHIQVLoKzWbO/SKXfniTpvHokwwqsEfNyIBRHE3rYTeEYO929FZc8bNtNRyynN71j6Pr93dDEUdFOS6rLJbjogQe4BjEiMfJBPPGk17vG/E3Zvx6NU3jgjQSogmhOCyp08tzUfrFEgqR+ZDIgNjeUQp3FfaF9bcclIaizKtyYUEMo7/fjK7vR2+89+9+fnOHpTbWrEqYOdZQnL04x5wg9osCb+cnePGLdRJ4xhhqXKVhFUu8XFzgLZGP3izi99Nt7+2H1/fe9evbN3d98X70P97bdx9GtNAZJp6e9K5H7z5476nBppazM/y/7PNaF/h/eeL0ej1IRQQLGSztxF/JoWAWpcuhZlMoSz+KuRlwLMsRx6/Ez2kihz2Bj+Kx62eZTEKbQSgG2+nSqWc7Do/N8igp7Zk13hzdvL67OxLRDAsJGRdSHL19/e7D0XYiNgRiKzZq5tZyc6wcZTaAzeKqWFzd55Ws8ZZJQcodyqywdxDTGqK+oDI9biUHBYb2RbackyOybQvGnq0tYi0/YB3bymBsfkGN+km1rv08Tx+5WT9y+6Kaz2HJMz+Q3qKaUv9uk+MorOhT5uv2pUUVOLrqyVMit/HlNAPlp0BmpXjHI0ZYPO8Cae3TzavEHsM6Xcm2409jyMQ6XjHiUUZfUVKUfhzT4/FHi9kx6Ss16DAYJgnuByX55LSw+a82KLIf8b/M8R3Ww/18iFZR2fhoN0790GMTgpjg5op0JQUDE3ZVICbUgSNOAz8WIZwhiHBccmQsNrJ9uMRaqjVg4Rf0rGQLbYJhKrAtb3IJx5Lwa5pHc+gwJhgI9RRzzSaMif3VNPSHYlwq1EhXaDrQxzLl2KJVrAk183oTIb4R5TqD/UTzBOFkDLPGE/nuieYlw/aixCuCNJNKX5mT5BAnwy8nVJG0g7RNIvtGHP/FD0dMhW+YPiYMn5oQZZ+wMGUhhGAWtjTsGEA9eDGjN68G3ftD0iWC37y+f303uodqg3MguhW08i6WRlb4MUWQNZplASEMrb453zmkHxRcIPsdHGH3gBzIMk2Okzgb3P336HiKCLPw/gvBeRbJkKyIlh0gKpTHJ/Q5Tmf8cOoi4fhYydLqs6J4pCNXNZKWQiOcYdUsdAlhT4+3CRmnUc7R3+/d1RKpgk0JDPIBtlIiCWzw0tpo6+E1lzgt8Eo02Zbl/iONEpviuxvC1cGaHfGtsH5LLNbxnJQ5nLll6pFCUigN0jyE79tx3w0DRM0AQYSLIoujcig2sUzscOZsBRxkQSqzMVDaWrVreaDJa6+QOl+w973IPSVn/gMWQAg+JjVngxGrCn+mSKEo1+IMTkrKLAt/hSCfizmS0aKO3jMowAIJYC1PjNPs/2PuRSFyxQkS211hE12g6v2d9x93v/ys5MfMdpyxcrHQGy8CDyd6HaCiPZCimzptxV4Vgy2msmGK4Fw4rODXhU1UIrSfvTg9ObtwoFEapSsFGAHSYubzG/hPCzcRPgZjn/DiO4z/Im4QCbejm1/uvlQtWcWwcOMyW8udYq7mCj0yloxka6Wwfuo6YPaG6S/ADnCFBjqdfuSZZZRUsg3BJ1iPklYX6dzMC1SGarezdqOpNY/IhC1mI0fN449VxGaNRyRkmH41jdPpMGkGEB4WCWVRllkxHAwAA47FRRUy2BB5W1dBhQxsRnonBHfJm6nFQ6HmEh83B0gQx6Bu6J7MtpS/7KdMqsQqEj8rFilYycFjaASiPlcmKH8QypISfLr4M/rxWQ6HgIXuEzWUsEKDDd9BCPBqRrY+g63V4bHWPJ331VTY5a4/pxBtGKA1Ud6/nuCRd8UsFyNgrfjyiuh3KQbiVL5QWHH1tl+G2Jo5Dgc8dLSIcaIQIbdcTUk41Omu/MwGbX0V/HfUltnw7ZU4PaDMlCVT/3bAJq2mU3qMFZAdr6ZD93S2FT/9IOxntMDZU4OvkCXMjVJHZ4pV4vHrvkYd0h+jtK31yKiniLnF+R/wzedusAr1+pyRqkSJAtAV9XIeQ547Vk779tf7H72b1/c/6oGSQvAm31Gaocjb0EiibH0/8n4qOwfCUqzgpV1eAaWfEQlcDotxlEi4LWfbUw6PAsl4d7l2LUYcPi8X/3JFyLlzZAS7w52JGTdMiej5IbgbUT6NjJsiYVsb9zlTBhpNzKAZCBksu76K4ehGSxjNMEts8DYenk+2OmdhTSPmcq7AtsXFs0eFs9LDQgE1kwdLDeLquqCUTA106gyvwYSCGmJ+ay+aSjX/mKtzLmyE/Fj5sdpR4eIcpe6L88H5yeBkcHExOCXtR6Fjq4WYj8u+OOEFHpjheH0gnnerdReJ04okxvsuRbWqITz4ccWiJAjn5999JYOaIiFhJdHhWu1UeSCI03vmStdLx/5Uqrp8x77+ijLX63nm5sYXanXiBasMKyee0u+TWtcpySEzpBKizQVqPTdKNdphAPZ7hjHp+NhaYHkto65XVWjsudVHgp0rBehmB8gaKRrZDyTRRwd1nrCjokbAftAhSGmC0fHY6bBps8yNEGES+0E1GS0KrD8tsMoxLfISkeb4e6eLia5gCaEuTYouYuseXQaPXT8M972E4STI1klrtkMBQauEWulz11kc2PYkl6HkCsH2DQwO+w8WwrYGrrU55A4AabwK5TDa1RABNYAma6Wg0liGvZLlIg33MteDNbSfz7FkIeuw8TN4WmSoRnt/MAaZ8aVFpQFXI3Wlv7VEdyZ4cCArP18/F5vw9suv96Yh1sanne4Br4Bc84efzi4F7QoWgyqJQM234ubm1nLM3QIa2us6lxopSoaGvH+K3GWPx8hdkOUoVSerdk92/M08LbtlEcHrFEOGWzKG1T5Ho7HncroeR2eO0Bti29he9ckRYO3xajJeTvrsOPjRUbt87OYUNcptUIMaT4a2VIUIzWFjJdtsoMFGDXCviAOdYJuieEVUERr1big6YD0DPVDRpaIvk9LEX35rwi0F6UJsuLGOul8lzLCmBKA3Qg0sRSAjqOcctfKMI6jyAjP4AS4+l3JdxyM98Inkjp7+GcmdOcblvUHK0vXg3E+WHuu7N12dXeq6g/Kf4tn499mUUIUzMjKqtFRVomoRfZRBxwbGYYatm5WKlikKGd2CcdedyqfZTFR8bOvb8JPhDJDdyE9mUTOfQdUrNWKeon4DHFuROt6tdGAKgNYpnUnd5tXw8zVxvqQI2eWpDUh9qqVQDEPDVx7VS5IEi8AyPjHiMjkfc2OYz1XMqF1FfPKXL3dKd4xEl8onbGtISeFpB7SmIWMqAEEttRcDubk+h8j24irvi9AQztuas5J9OFP4nGWbdtLOnFaEcbPt26UsCPgkISBHWe4kI0EwthJk9nthmzpmsV96q6hASjFXY4hEm/KkfDketoc7MLJ/hQCdPQjQvgUfBz0BRlOMybPu5MYh6iOifT4A/MwiTaAd6zLNNsutAs6HOzWOyw5qCdX88KFEcmfL8LfkGcdDnkYo56Iqb1+QngtCYIisYetY3U3IzRGJ4mj48uy77eYoORq+usB3M0dofijvcXaCpfOwwKjzZhjxpjvs9ERpkRpmdfetUKuw44DnkVAwrRNNmTDc263hbRqFXzAGhhNGEY+m2Kn1XDXvyVL37RGOwKPpThgkcfwgUO74QrA0/xhSqCUjbGYjeao0E5tWF7cOOddT2mDYXxfhPBm657MudFajBvB0rc/XSfYsFrL5zXJIoZBgzo66anc02Q6gAVdPdLZr7mn1tiksE1VQnrZiCiN/fjgfoZ7nkpGx1STGk7FFrtKa7NaqxJqa6qF4WjW1aYARwr686Av3xcUZytYdi2S72vEYV1dMhm7XS7X9lFxQcnZIQ7b9djuAEg1AOToE5aip+A3KSKfSMl2lVaEJO2hMbVoq7NPvniDpkAtr6DrQ2SXqgHYfouzQMIOs+fKwJtTXOzgW1pwpnkxO260d0mab1XXPhSKQL9HMEJfcMrbIL3LUVRsnlk5hTVXmY8eDAGmL6AtAbneFyNIxLbMQ0Nezk8HZ2eDsknZQLk4Gl9pzqtz2s1dfzE0ltaMdOmrfPlRH/Fa3INlYdACtsivildNXVQ+HWPj8XHcFihm5dsC7nne7/TqZMWD6EWpUwYSpbXKZq+xXbWpx6WwXxi41yBzWHqYYW8Qiur2gNNh6mchHVv9X1u5xtHGPw5hPq+i59aWPvXmqw4vlg4ytLjuNeyNW9yLJoasluy3GXRT9sB03ONVH2MwbTgIPHhwspcxUIakvBsAUmvsBbY1gWdYPxF69KdnhO1RQ+sGC98OoUKVrBE/kBY6+EqXqCyoM6J26j/U2Y8B12lx2D/r+OdWJYo1Wm3qGOmxRXTSyObH4/yhWWCg7lUp9r2GjfFuh7xglaSjrZxnOm+dCBvzo1h18fk1NTVnT38scv+hjQTYdMMKCDzBbdPAupExMqpB5ElGbLd240ApCeQudM/POQ1/UdohyGikG7Yco3WIq/0oZVu/PfXERNj6dPFN1EepXnPmMVUY70aXQ0eCIpm896iFr8BSxFLlar/BMeUL7HFhAV07Myg46z56DcnTEmCcV2T50rEaeYN0FAgVnZeIS4qkDq90ZWkcPlE/oVDrLfVTczZuWvYWVRpsjdcvBkVrFJ26VUaliy7FFLfp4RvIu0hMQiMFjkM4Kaicom8EPCvmJGRxUPEs0II30drdgZV+6Z1vU+qUn7ubHpolIaaBowG/LKYw7/x1ZjLoiMl2XsrDpHigdT9OdmcK8KTJ3XKRTwNR2zFxn7WUfeWNfk90piwu2ryfMY6xPckB5l0gOsoDZCbl7ciJXMRlz7wHloHKfwSAzagOt2qUvWqFYL1UwfWW1M9rYq0OsSl2KrvxAGNO+Lx/gRo5tHzd+J1dvW0/E3f6hkNzBxbz+6ezU17cH05f6DkwTuZpzf2Fv6FArHx8pIzsyTh+bFKs569oqsMXnq3ANafjq36gkJ73G8/d4ZmPRz8WAh/GbhpZWJXv0nXo7/9P1tkEVI4N3hQ4Vi53O7+mInPoVik/3k/M61H02MwsJUDImNNgBtCHWcGUkU+knXh1xDUc2ODSujsaGG+uOOxx+u8GaQ/+Oe6OIq2Jw3cuK2wbiupltTacZeFYYPb3DCShPdz67N3pY/4y7szUCtXfepaqtctIHY4yizQRjdCoKzQ0LXoJ2KgissGmrXxsA7VXUFYvCpHvAPTfvkfNvAehyE4u43uvqZrxDY58JToLoV75iX8OFoOO74cvTy+0GBJGPgQ5ebqERXA3z+8tLw6zGlmdypz6ub3E3KwiM/zqlVJqR+/LjofjPR5mcH4/4lx+oMNtDPVk3Hbj11wxnj317+7Y9HHfFXZw+kle7vvlV2IsqCcFIzdm0KjCBX3gH2/l3PM5IijoD3v3lSbcseObEsP6lQ5oHi04LJf9Mjkeb51wb5H/mjDGUD1FAFZMVVKFvqYIfi7n06kaF5z/4UUwHTeAZRyXbQmw2Bk79YCnBARfNBydYQVZ1d+YaOejlWb35SV8mzd3r19c/jrw3725VZjz66QePm5TqgGQ2ZQxs5HbLjbaCc2WCA6qaTArNhI1R4fIsFxFPxu7Cj2d2c5SqfzqUGxez9cKfu3WtJ9OBavvrkL2jOVJFyyFAnCKRftDOyODm5lYd61EeU/pB+aUHwWo7UB2iUMgm+OrrW7VP+DXPQ/fOXo0j5M6Z44G5WgV+S1b+Jzr+NM4e924MNSed9iw7fTGYZednbJ1amsHCT+ZSXcUNUvodTxFH80UZrx3Tv/GZrL3Hkf1L3AL9K/zzP9mfOZf9/KmuU18imEUkZrVPoNzNtLm31/DgCh+1M4YkDR8Da/WblnRZ/5qFd+XUT4t6+5f2/sxPW7S/pl1IvquEuk+dLLVLdUT2+sMH3SxotdEby1zPuvvlp1HdT2ujv+vol73e/wHOwkbo", "reference/groundtruth.jsonl": "eNrlfWlzK8eR4Pf9FQ59tsm6j42NjZCfrLFiJFsjaezZ2JjoqCOLbD8QgADwHZqY/75ZDZCPQDXQ1WCDpGPt0CPQXajOzszKq7Iy/+urdr7euHmApo1f/c/ffbX8vIH15g8RPjTN7jMlRNKvfv+7r1awXOyPud5+zDe9W0MTFnd37SaPcU5Kq2XQkScVbFJRJ2VcCEFQ73VwMRHw1uWfpnYGa/zR//1qvQrXu8dezxY3N+385mr5+av/xEH5WjNr30PzODxfnsPH/Qvrz3d+Meu+/tdXS7e5zdAcmReH/3rvZnN3B3nU/7pbxPsZ/O98/X077/CxvZSvzNo5zPP7E/wC89jsXZgv4g6F8/vZDC+08+Zm5Zb58cnN1vDfv//dWHi+X9y8c8vN/Qr+7OZxBqungIWZW6/34OKcHkLGpbg4bN+2n/KfqzADtwdhup+HTbuY7wEpRAGkELwKyP/87//xX3Xsakaza3TWMS+NAK+dChosE54pRiGSGB0ER5z2cIJd7+ftJn+YnF+fTnxAhF/w8jt8kW93mL5a3c8fXvAkFTgpWYXZaanApRpLBR6o5MIKEXzkAoxDWREsaGWoIdQkFyEF6+hxKty51fvr9WZ1HzJPricnRs/8BzS5gU1zP1+68B5ik4evh6khTUENxZ+1cIfhXG8WK+gAHIbP8AI+S6fkFolLbSyzSAPKERu9sIkLY1RKlCfutWS4ZIkPURPFQzzOLGErwCZnkifzHiD9T/OAKIvftluVclyWCyYLMSmnRbnio6Vk8IYZ8KCJF8lxGaJM1mkAwgTTjHENznN5Yn3iXIix6TH+OO0Bwuf3d83dIrxv8Ha4bdzqZng5KnaIej2pgpLa6LGYt1KCFIxl5ea8k1pD8A6So1JptK60RkPLJ3Mc8/lpq830EvHJvG/CnDoOzx8RpT91d6+azaL5x3pL95O8QDsDag88RtjLAXgV2zXKBrjzM1Qn258OA60KoKmRLwB0Wi3u6vDKSIlXdSm8bgfssLcFcg2r1s3a35zfl8T9Nist9LNgalKRYIgdKxKyHDCKO8VFEkQqAkwSKSPVljGC3hVKBw+nPKylW2/At/PJZcLTiQ9oEVbgNtDkObtRw9K4UIQTmx5GkLGo1zxIr5PwlDpDwWqadES3Fq1UzYAoZ9D0IEmc0oPz1N5cN02L1n3T7BC1j8PN7az1F1CVfU9+fbk9DNeP3cAfZ/fokv7g5u4GVle4kDfdL3A1N3mKPKTC4lYF9NzY14Af3YUHsLc4rgDe2lIc8dcAvr3LEvUB/mE5yguhL5R6FuBPVsmb4OLj8KAb/+s9NN2PBiksSvYUkyocRe1oGxSMVIob7wj6WiFJDz5R/AccZ1GGpE0Eo/0JqYd/bhcXUDeP0xZMu/D/+KH9hJoIPwwbJbKIijD5PLupArTTjiATZWyAmEvDdIVO0/180+D3pk2orQG91mH8qRJ/6kKw/tAt5asHiVUhe0qHWtBLwjbg4PPCy5RSXYiuLrxHgX0VFrMZhGE5rcoYrdJkUvGDTtTo4KBTyoBTKTHhSTCJEGFN4FSIoLh3QSoaLZNvUvygmkflUyX8mS5XkZk0NqsZkaOjbaATdV74JA0IzUFGnYw24JkX1kYdwfnI7HHso6V2v1zMZ58nJ8DezG/CDDgFUbO+XeQ37HZNGvfIAKfFlypiUmJaluCjo/XBGh9EiDwqoxNYzy2LBiBEi+4osrGlUSSi3taeyb/jrYd9k2qJyHUhEJ8nq08A2Nw5RMCnPOKBS652fwfhtAUfUyIvBejZu0+MFDYN4/xSYIZbCO+bfDNk/kWfZQZ3MN+sm02OAe3CQjC8dcmsKk2xSSPDmjMxdhVS66I0xnLJg04yMa9kCMzhuhTKa2fByphObYN0e0XwabmC9bq9gH7smf9NCOlhuOaLTZMHVISJC+uSKnlh6H7IWxqw+jq65SZ74zmW0G7grsIaprqMEGs5KSujgTvawVTACQHNbIycW6lACmGiyVFM4qkUxAuQ/EREc/2+XS4vkTTydOIj0eWt8Oukd2WcnvVEocjEVLDjg5tJO66ENs6DjpEmZy33lMrIEGDlHAEuwZg3TYXOABsmQOmEMfG8jJl6EIObzYYhFLoMSehLQXhZOVzLs5KN3pgWNDJKAxNOaUONQH5FAaRtVFFJbpIMVLvo7VvKBfj2m10C1R/buVt9rg9lcKnLdAwzLQn0eBKgdDAyEh8DRaNTaMkl4SrQlBA6L4EGIaiQbynhr0xiqyeCEGWI1rxYsh2iYJ73L6GCW0olI18O0Lw/M4MPMCzqkIGKCB3Xk7K1NaNtEqeBmyCDopYZwrljyN+WyuABTHDOoIniRCSndlkvs5F3fKvhQ7tuh60PWUa7ZeUWfCW+Dbejd7UV2hzBeBcMpyGagMaI5J4YSGiUKB+NSCSQBDVRvt//7mWiDYNB6nb+Dwg7o6TZKtRH137r6PfdGqYgKxQBKrvLxtv3XuUBqt6XObxZwZCFfy01u8zrvMsB+YO36YL0va+yd2fwPUwZ3jCMvtx73AEOjv0Mtndr+E3KuJeR9C0GvqjgpUP7nKB5dBvXNJ/cauU+/4HZw4B5vn29vdvnQgUBLIuwpL3gRDGrUrJee0mdS4SiY+uiogc7pdvprsNiBdcf3KrNuUmTyKpjEx/g362bbR5li/eb/IbDaDe2dE/UVGjnxI5CO0W7NabkAktOWB61t45o4Y0IySA1iNNEqUPP9Sl28F3h0xfT5kUocvDM14mLVQL0o5tHt/4uD9iFnU7vclLGCgahnOiXg6/eoqc90QB6TjjgfFjj5vOyQtaxcjeIsjOcj+cgtftpFVZL/wOvvSSw61tXhVXVA+gZxzCeg9UREVxWJkbiynrBlX+1Wbn5erlYV+CWE9YD7MviFrXIqgax2xDstCLrhNbtoPzbw/2w2I44DaJlJataOZm5w+nhec8BvUtNEtFa9N00pcG6ANGikaOIsC7ZxEyUEJQ6rndxLo9vMrlufTLv4a7g9k7jP+OLLFaxJu2yMDEllZPhnI8zMZUNGggFSZV2nHIdmJHBSxKp1YZJ5r3xwafjOM/zo1fwkqbO/iOfkuOb3R28Nt+0NTn5ypbJpNqSSy3TxxVaDSGVtmdTUNHJOEZpPYZjIOGKdNoblQxiCm1jmiAkiq6I9MY4NH+U5xSqOOaF2eIOVjc1oc6eZEk14Rq1lo/BuAmBi+AFZd4ZCz44ylIwImnlLIh86NEk6sNpjHdfe1bpJanx9JmH9Pi6u9fON4AIrDjAwUXP8VHJ2bNW6jDL1AMorSrsaUUYn4ptBNHj3FjriTGKK/Bd9QTpvHaUCOWC6o6XGMs1jSfc2I/Q3txuIE7OG08nPsD73x9uNev7u2aRmu3YdYXBVQSZqCTTId+KMch3SoTEohQarRnvoyPWelzFgnjtgRpDoo/88FTV0fXzCitzs2juM0q6A+i7dVJhUpYVAahVejIyFNUABpQVkSlGcEaySEwKxAiZrBRe2wjegkWzMhnDj5Nh7uaL5XpyGnyZtoxefvbQZBw1i/uK5I8yH32ywJlQzI5iekEhBu2TREctsXygQ1vBAwnCBBTVnjJC7ClF1SnoyZH9OOsBrrvrjdtsVsPiRZRWmGQTYtqMMsIi98o5FY23VsdE0fqSiGdCIyGMMUeSZUlxd9JVWt5v3GaqTL0Tcx9g/eMt1ETiNS+dU8OmE+hqXFBYUkjMKRUEEIMuKirTyDzhwGPy3Cr8wA3w8NYEevP9InSxgbGBodIvonay0ICwapQc51oyBtZSFCmAWCYuUidRdLsEiHXwqFGllOQI9mM7v3n0yiYW5uXcRab4ur2ZQ/wuW5Kweof4W11FyBU8KgrsyLLAzmTaVAk1KkBjJI0O9SZBVhfI+pIwGnjyYEjwwVKmAdC0jP/MUqdnA9AIMRnGpR2lUZX3zCNni5z6FJSJOgSH/iCa9JRwJWOuXkOEeYsYb2C+zkdT5veoZ9uQDco0W+Dr1tRKoLY8aIciaTpCaDaK9UOgBvHNfFKAfM7Bmi4VzfPgnbYBVUKg1PKThLi7DA3uetGfC0iF2/v5+3X7W8VpcEbKYDAjfEKECzFu7zuoEFDigzTKKSqN90oqb9DIYehIheiYRJf4OMJXi9lsqjzAI/Mec55+2g1pGtS2NbsFTJfIN2wy3FtuRjG7AEEi4j4w4wGFPth8LJIEMFa6oJP1VFgm/smCwuuPbtnE9q6mroMpI4AXjAq/zgZ9TZgahXbHHA9jh6NgvNipl4K/BJzdRt4IQMuUeHlGUtK5gFaksZUOp3yh7cPx2GTGlGcgjBEvC28NmOWZQUteCMwugbnJ9yr0sSWqhJTKl4G0uzoMIiurb9npgiLKWjZGbQmJfiGlApjXnmlveLCKGh+I9Fy55IJhaMWFc9RWx16wfg2t1VUmqmFublVZYIwSdeEdkhXUg4jg9IBo2PMzOHpd720Cxw/3s03b8ffVe4Dl9hzB8ArUpe+t2WQ2GU41yhPkzmkZTCTECceicCFREg1JWnnLcyFsr5Ty+p/Z99aly2f08yTeacBe46DaARdwPi4KGXkuEQdWov1CpGWSEuQAnlx0jmcUEuc8f0te0buFW61hfhUW820l4eF8cVNg3JrJEh40H7f9LmiCpBLl6P2DtSLXpSBSRYEeLUnoGjkRiDKpLh34pXJ+D0Ufjjoj61LyMhwmn3WaCud5PN3TfRbyMEvsYcj19mNfGVsrTd7djmCCdCxHChQFyh3XghFnlAmG7EJtX0iynQ5nXaECQJa8jq1DkO8eNEdxH4FfQtgsVscG3G/a2dFff1xl538Smg9BXhzl+GZ7/2p74qTO4qRleVJ6RnrFIBb3Dvq18/eIo+6kWYOgtjfzKtOT2tLbYOJFYL3teijsgM17h01VtjVjpTnP9fMhfsqCrxNSqAbp+0XAH/8t03qxo3lFWLKsYnWGE3Ricb4dvB2F6ZvF5u/bezmo+8HN7muWc8lwVMhJJbciYqzkpiClAyUUE5Gh8E5gvOIuaqstqn0ABgoST72SuyvBg6gqttse7i86mCeqTj700L7wzvrddvRVs+2LkEVD4+bzxdYEbeY1m37oD/VtQvFzOWwPLW/AAu5hJDXaBKCRKmohWcq9RVtAKaEjI5z4GFMSBv0kH5Pi/YzUXzY5n228CulmOt555SLJtfi3mozFvyAhsJSiNkZbaanUziWhLBC0zSQEtJV1ksycXsjr9q6dudVF1uuTuQ+Q/vPuzup+XhFUKVvcWD4l9hU3aiz2tRReGJ1ENJYBj/iZeZWCFso4TaMXwGmgJ7nfrW7ucwGzQoY+vblu7nZVmvtG9di/3T8dtItlRua0srgX8r1N53duNsuS+OuHMfWlXctCBsjPzxC6p/C4B/MDrOvHitguxmaZ3ebVDo0VrlpPMTzxTOiPWXK5RDraLJsdcLnLxMOLDi8oeq5ZMh7OFzfmhvC2zKfHFgHQKdstj2ELmJZBdnE2W/Yuz94D9Cgcq2HkPcmRyk4qJHeqZIyQZJo745lUjAilA5qeyijnTGQOJaQ3gbCgzWHNlaeIgk9L9PZ29TT6BN3287Re/vEnvypr14G1P6Aiq7YUBs/j7acUeRMoOwHQj5+/395qYrsOC5SoOy4Z3BtW5d61ZZOuN0v42PUWnCGJcCe4D9yCQAMxKB2DtYZqz4NNaKsbp46vN5Q5k6+k3ZyHMu7Xe1h9bsLyvuLgUXGqh06Jak3MaOubBzSxHYsQeYzMSwveJcbBM2OF5F5GZjkSY4xoe1n51bTrpr2Zo5Mcu6nPEBVqUidIs8NzkMNkUEagKsm1LJ3SlgIjRBsfJBPJ6mAsUZwzbvvJ8NQrnAr3JzxNJEWbPjfrz+u65iHm7BrQlQg3xo5W6TSZIKLN+d4yUOVYoNEYGogUidCckcnAcy7G+z2dyXaIv717j0bdC3ozK7jBldSE9YemK5OQFncVFaZpaTJSerZa7cXMq+rVA3q8ekjFrTerxfJz0+w+/IEyS/aEye7G9e5v34keqnwCIp0WwhIbQXimkzRa4tUoIAoaQB3muz7Mi+8Hefv1eg3oNk6WX3l6+sIp7E4LVjg0ZbEnoSdEPif7m+rDyGfWhOCFNoBEIJZZVKHB5YoIqEdjUIHLpLRk/cjftHeQGwHCutuVnRTzPXP3NE78BUf93I26arZ16lfw632bdWtYzO7vKlw42VOibUqaHDSGGKaJgqiVjdJEY7khaOA4CkkyzwlRqGshRWfsYYDxEW+ZSbf/TkuP/XkPWxl01x+jI13awWaRaVAhs3VPiSsmpqQB31e4FTSQBFeDiYGA1B5MIipSqhi4ELh3EhikpOyRddEVRGnnDhF/7e/b2QbhTqu8F32oTSp+0TlKdxDb3PZx9bCfslOKq8ed8JqZNquRo5uFxzX4AVfT/vNquWqK55zkynpMv47eno6+T+HfAA7Nqb8ZdXeupiFVIeP0G3yXiiqVPaUO1Bt5k44a+Cb5jWqK25S1bd4KTUK7fZM6mtiywj55azTJb1SRclHaZ/QyNPkiid+uXOqH8btffvp5qGZlWa3mUgAeUxyHiEWt9LuslV7VN+JCjjTPqQ3BEKGp5tYHr7TXYIi1TCjPLUXrXBLGAfrNkHaBU4e2vb7d3M0mNQaLmQ/w/edffvj+qsvqqQgyllXrn1EBtwfrUttxWCcxWZGSJlZbRcGyvNlPGeHKMeGpkD5QHtwRA/xjWOf/3LK9/ojgLrsd+VkbkEfx8qRkGH7U4f5/d/P7xcfv80mAv7/7+erjYjWLWUIu208wq02xYkKWjUrtlESzemQYQTJJPBMC/1BQQWkacI0YIrlKFKLnkinCIu0nWq6Zvb7elqbbfJ6USOXUB0T5t4dbD+nJ9xnnFfv2qjy4rJ5xdrakgiB2pO9KnRCKRc1yrYTESQyGO6OtpzHmRJlgPAvU+GG/af1+V1iy+z4pRU4/5nDJvP/8Lt/f1hDp0l6HKWNUYUlYMqUqyUeHxlHGSYuOtjYeuI0ovqIlXlINNIKA3CPCpuScMAOqZDVR+4djE5cJS8sZ/PTzL98cVFPvafRsy4IhZ5sdA2AhQKeBKWs4KHVBYOpzfcoazYpeErA6I0DxM9FVuVo4GWkChHyojEjrWAIvBICwhFGjBV70zhJDaLLWqKOrJWWhHxbzOYTJF8zh3Ic+T+60UVE3sKz1qSdVHlzJcUjXaEMkl+tJgYmECSNdzm11ljrpmFPEEeWSogMi6te4vIyI2k18GP3P6Ks8CkHODDWfXHxHwMoZ8124FtlhtbhrcFjdfjMr60lujfQJOWPkckxOIEvQEJm2LiSLhp5UxKAaM0oyYXNVFGkZO2XcZd/Qba5DPIyBHt7e5h8i6i5gBhZA7B1g+ubnq20K2jYDsqLyZ5khd0bB/WeAeJXriuQDLYt5LlnazVCT5l9WOCKTgb1Hvze4RSokGbkb4XgSDp2ayEQuBB+TsIlRHrVGtwb/RkksysYwoI3cKl5GFe0mPpRAW5Jsa1/VtA0gPdtAZlLE87FmAEGPP9AYLDov3EsUNlYg5pPgMkHyDIK1+rAP0SF+YpvSRRD/MHHfZlw2mL/J95s8qkaUlKkTwkyL/rG7cIYEbqRwMglcAMp5owLh0UavTS4wrFOIhgf6tvj+HV4+7R0I2tN+hj/HEhiC56pZL2cVpw5N6UUZO+V2uLB2pFXoqQo6UhaCD9qzfNRJCW9sQrucGZl7NEAEeWQNzmN3xv2u/YSg4TeH/sjtHWzaMClLnH7MAUH+8s3Xj7d/yL+4ar6MR+26fl9hu/cU5Z0wlURTNTLyw1SkQhtlknEGLXZFUCcxGaJyRmoVGVAftTiSxrNNbLpr19OSZX/aV99tOQnOd/NbQC74ZhFwdDu/Wdc79bI8NinFhCaL5mNzWDRn0diA2pJQqXzOo2CEeM9VRJHuJQgmcOW6UwZ7rhS7cpvFan0BQ3x/8iMR2e/my/t8jip3Gm+aq12MfViXqvLoNpswnqEVGUkOHiBGbdGEpIxaShgjyYCKSeuA+pV41P7grDhFjskzvPanLWoHt5uc4IXYh18rloAuK3JpQZ/pUxyFbYXcczNvf4OY4ayFkepyaxSviSk5Y7/veIVrgc+nXWEXar1iyhh0JrQDIhWzBj8qbQKR5pTYbuf4rSsfMVWFpYH534ggH4Lrrs3ZcOuaQ4yWTlmyoOALo8dm5aLSzq0/s3jwLmgJNnDPEgiuPCUSHGE2qpDqTO/fl8bqLbg40fGxMXZwDoVt9et5dTLts630Ly/+6mw8CNKftzdGII33bMPYCQ1TY/TIjS+fK4MRUABaJa+8yR1pnIhUQZDUau1BKJ/U290YHki3sWW5E0Weq/rq9qvzC9fUeC4zgp5xaG+9vG3nn/4QF6Fpdp8pEXzPrfwy5nr7sS+oTImigVAGjIpEVYpGW4KGIQ/M8KiNBh7dYZrHdrpdyZr19cxt4NMkbHFs4sNaQu4X+I9fcrLTzD1WE8KH4o9cRbK16umpp6mZmBj7zmMNMYJD195ytDciCcGh3uG5bKmRyTIdI3BgTiYIvcTIiWExI+0GNoi1zZT06Ju7UCwbN1uMKHpeVixQEy+HXW/0MRRwgaGotDYyylzSEamYgzU2gLJMBcUDpxyJ0ksBRAyaQ50D24G2K9M2JSFOPOKAHjkprdmui54fVJzpLyu1solXiNJ8tLjiMjKnQtBMJYv0AYPEcRp1G9XMIsWcwdv94iq2q2ymfoD19WJz+6jrH1A7/9CuFvN8Fu/abWslrq83i7BZARwfGRazWVcybm/sVOQ+AvFenHsRfumemvd78tmTTU0Zjp7TJmJ8km895l7HxDsPvgeMrmC9mH2oOJpcBjrOsJAngfWqwd/lg2Db3e7d7ypOH5X9J6ia4hX6l0f/S7x7GHu1ra6Iy+Rm10HGQ1Xmek8Nev7G3uKq+ehm75tYSRlWnnt/To5pjxhGbUfGSmEGkqggnRdS5wY4gnsvNHjpTTDgbYySRsZjvxRe3Lm8PxD2herj5eVy/0YOM1yH5O7a2edJZesBHK8qok4B8w2kHAJH1vhxm26w3duvNncZKQvDM0rYs2FdLt8U6nrB+frnX76LuIbb1HZ919Zh1XpottVb7yvqkquyBKmqNITOAfYvnZj4fkvaruBnG4c3LHkh+QznF4Px3a0bDaFW5eE7fRkIn7te0ORXZSk4c7Y+PBRgr71gBuD5gL5C+txsl8q2wNtdTd1SXawTzSdVVFqYsYrK2hzKD8CU9onEJLSlWhMvlBWcepo8ITHEeNSdc/ebRX7CpNVaBuYvyv/u9kJ/fpBZ3Xb5QDyMlFlxlFh6tjX0LFivECkISb3QpUT01J/R7JXAz0IOf1oBtu7BunktsHepZ2PQbl6Da77dwYLv0ZXCycH20aD3nSWmWk4rgaQebSpTliJKH5tQ6iQHyunAJaEpJuqsk54ZQjmzvRKok9S7ovRTSp7DeQ/o8Yj1rReJvDUczCtrDtW2K6tFvqXj/RTtAiURCMp7DYxrFjmNXkphQGZKqGiZksfF/9wtFzNYzC8m/3sf8OrxkkrAmvX7dtncQbZYhwOKpqe+7rQGgjXjtz84YUZAQjbINfSI8FSrxKgXjAKLKQWVa9Oz0xF3fKP3XeGdi8Tc92Z/bd6ogqorg/+nTyjI526WG2es/7j92dW2PNHmdgUubr9UFIwq42vPyOvqYZx8rHks42grtdA+5r3U5HXUVEfpfQoscE7RW7CgUMLwXsZxy+WsDU/bUu1uzHKPDLiItCmf+ap8dBqcn7tBV1s0tNTMK7ZXyy3380McPXQ4PF82r0gkpqQnzK4mZV22f4qohnVRsIUQTSIQHDGahKSSSXmHiwuUec6gJLRev1mZN0a6NPertiLtv2zOQ+mkZBJyNJmUcU4kzyzomEAyqoUFClQkLUnULjnvuTB00HfNp/Lw4kR9UYYecECsnCS4is3jkIqDd2Xr5UlJIceHEWLuNiKDzMUAU0qKc2A6IvqNoolFpoEmb7UaFUb4fS8u27vlYjVVSfFxbti2ddgXHyx7vQv/D/QRdjbeuqYNQV8ejprCgXyKmbdgIVfAlTG47cFWi0AmCz+KT5uYYhSj451YZU2gQiiaD59y5HsRNQeVvJMop6gUYLjnpzVGzki4Pq+EXrXe6HnGAU3+FT77hVvFXx4GDRRUoGWp9SnJYRkfbXtKpVNiTFPlApPGW4iRBaopjVopLwRxgXhKj8cUcPrUwiyuJ48q7M1cxqm+zfceUY+LZtNHhtoWetvjehPSQo5WDURRopPhJjDjvApOdT5AYMQAldbqFDxDo+/kVujy8+Z2oqTxozMXzQf+YwVpGyfsjjJ/wq8VrT5IKZ/YxEQwo+WTgUAUWEd9RPEW0BnjPp+PFAYItyEE7TlSw7+lINtOsz7ZZhnemCrbsQo1qXFk+XhPGIgMkNAbRkeYceeplxq/otXqVATIp7LRreCkF/nhLl7/et+G9wjZalL0lzO/ttEwCJFbo79SVfKgPPjI2aQhESvU6FVocalp71E35cRVJ9BUiIkoHcAprqUg6HHS6GKlKPz97+ot6Eut4Qox+uNqsYTVUH66KfN4zPk7yiPA6qz4LwH9JUr59lNF6a6yHvhL7L49bv+0s5xEXGs1q7LSmKHmwuA+YPgJ2MHN8z5h97U2Ls5Uz0FJpcTLQ5+7zj0mnDbb4zAV4Gvek6FGn5WccFxrtutubd0BMn6syJ4rE5+kvBBouNBq9ogNL+sfqUm3KXM/ztGyGyKRIhcWgpSi9DQSJRiwyIQzhhBubARm09szYx8F3a4zd/2WsSl3jM0zKq6tQ/u+3fxhBm41R1o8+faHvK++fxThyd3rp1/6SKMl49ZIzy2ECFzZaDiXLOpchD8Jwr0i7rDs0/r9dnJcKas2rK/D7H6Ni/x6fY/Y+tCuYZpqIHWPOSBaWnycvYdctH02W3xcI64WFeTSpqdHhb0Muex+caIx5FIal46nJigDIIRkuqsGQcEoI5WMUXApnE/95Eqw1dMoxVeuw8H1dGd5hp9xGBFc3M83f+uymdvfUENsj/JCU++027Lovn3OqbcTRKNSn020GJXlijoSbVABVxmJjEvuCfFGAQG0XmmuyddPtPxqbtVlyc2uZ4ubdj1VZZXBJxRl+BY3uWtUt6Ta+U1dry9dZi3ZC60sJg/26EcQyacgnETTiHF061XyMdjkCS4wmXKXGBBGkbCdvyRSnmxSmjxMeNg2YLaYVxT8JmWhhosgnFN2vuZJxCMjOBMstTJSr4ARnojxyhp0+5kXTAVnj2iezKzNGmbb0gDX2+JL02qdo4843E9GidWlgMd//XYxGyhMJUu7oLY+xZ7V+EwgdyUGO0Ql/F7hBrEiDVxpdiG24mevY0dQ0kYnDKQQkJuEdMEB1YRYKXFpU62y3vT9bPWlh3LuSNa1Tl3Bpv3NTVZ3o/pJhzsLf0QEfvMwqvNiK1IEaLlhZuyFaCbYuTSjAU1j44nLBYA5yxtAmsSYtCBojQogLCiV1DFR0H7KFsf0Mvhw4p7maD9sh2Ri5OhHbEMFUWxZUlRfxjPgnJ1ttXDtCSgVjEbrEjUhZ84GTYzyucU1mizWcEMPi8L12hS3936qPeeT0x+W1sjXf4KbFS6yRd2CYWViOycXoo3Yr/w/ijboSAdcGTGAZmAk8SmhfWKsB8eDjinJxFMRDN0hD+ZruMut5lq0smE9rdbsm/ywDNt6MevE3LfdiLE154U+X12Oh66+RBwtw5rU8ovwjiD7bW5GuZAun6NUgRseoxHaUQ+aEu4iOjmJapGA8ZSO8E4+V5obMC9W07LNwbxl53e80WRHsqK3sCr9DX0hMhh9tp0iteAQO0+DJx8McEl9dJ4qqWhCd8ThOnb0rTuF3+9u7eQsUuPd3+psE9uTGMfoZSxKgf8/W9hGa7WyOuVELQUoaZkBZ1PkUnPhlTPeUcqhn1K7Dsfr28XHZleqbD0poY49oK9YfYTlGsVZWgy7kGW7iMvYjeIZajAK61Si3CWWGP7PRMGJC0rkQisSLV2uDSGR9FPmIZq4WCL7rptJidIz9wE9/vrjL9+9G2jzxnoMRX227huGCTG7vN9AsxuxnXKYU8qjF5ZdEsrtiAcoPw2n4MpyB09xcUEQEahmN2o4PlY2sjGWXGatKWbPXWvCRdRW1kulUCAKnhT62MQ670JCx8ABjzqhMDzio93P8MXvN7nY66Qu2v68B2T4Id/9a3e3Sz/tjukPdBUqS+Rzc/6aOw1fk13Id7cun6as0Ji8bAMqLhRJFfrgPNwYTvGKae6NFoI5IbiOxiMOObomhDEHVPEkUkwDzklzi3ZFfo3Y5g1vv1igmTG/uS6uXMZ9GfX4nsjAn/Hn/7Ib+8eHobsG6vik2edmvVkslzmM3u1QDQsKUSavC34h8qNvfy75CQoIFkikRlJmnORB6pwkxz2TXqHJDJZ4b/4J4rovmMM1FrSmy0lucgyx4ogb76nQoswlOIdJSs4OAybraEhEs+hs1FzLqALL0kIZpwg3nnLPKUmnt2B+X+5FPkHoS23UZBHwJ1z0d10Fydz0s425p3I8aLLXv9LLCpKKnG9SnUDDq3N8PWw/w7by1TaruH6TuCxPZF8e3DGb2mUBcyIus1Z3vSLOWassGKW9SYYx8NopIxKJMgguLaDeV0CDDQnIaac4nwGa3ijsnf1QgOZOA+gKN0s3j27dlTZEO6wmPZZMWVn0FHmsPlsJUxJ9oIL5SFH5Ju0jNwCBuUCElywA6mdCLJzOE3mqiODX+1zoys0uki1y9ElvUUCdhPDnx5vfbn/5IAjqgmKmrMhn2RsAvgsf+UzyjsAP81blVjHbc+zsIhFZy8z5+12Q2x7mQx3JSsD/tFFMKhONxuViWOLaAD3sR3oYE3jf3AFK+stEk55OfkC53a2KBhll5d1ntMdA82Y5W2xmrW+aL5+7SMOe7Ppy7/rLx76InpKoNxzLu7+GIhUotRZUNNQmG7kkCpLQ+qBG9f601+4TrPf08sH9TRveP24YdveWs2azWMze57Lsd3kYj90sfJo8xiH49hTT13i3a8uS9denWVtRTbmnGw5nZnyA63w4P1fBKcu6W/ycZPRT9Nzfmwid+T1fzHOWxf3MVaj3nk5PeE2eB+RpxjosfphvfPNIdx4rqpyXgc0zutw9C9TPdaD2lD3UZ7RQeBaov9WBqstjiPoZcbYjMpIRsa+samWkR30kqAxUgufWE6Ycsdoxo2R0Ou8lJpql6EkZiY7pDVxCuD1OfCgvvss3tv2mutSzbmBFKmPZIcaI6WmhtDqHFgqNhJRDnEmK4LQ2xBGhGeEBqeAcjd7EIAU7SYuPbUTran0JajyZuiiC4+ZfrDtcHffLBnBwsz3pUBFVkn1RJT45aSjaYOeQJpEuDpmC5twmZcBaBlFxBwJJE7QKUVB3mCNzgL8cs7kIYR4nPuaXbmD4IJ0sDevatjRjKMCZtedQgOdDXi4EooJNLMYQXVAOuM3WnHM2eQAL3J6kwKTVqk7MfVhZJmy7HXyqqcMkdU9dSzM9HQRlZ66EXDjM+eiS8JYwIwNIFFcJovfGBdBU2sOKDge4QhSG28ushSdTHx7Ayneu4sp9rDCFXmQtCFS359CAimCVl5SAB+8ESVJzkJTh0vLUapW49IoROEkDhDC8z18uQYW9yV8t7DICqMd7FZl5Pc2gp2cNKs4Sky6hfqKQy0FGw0wiMqQYiUgMpEWP13vhmPY1Pm/+93Ie68PsPQb2VZaWH2C1aeKnGtOh7KXBCJueIlKTcyiCNCAEDQUUlVbmwq6G4fK1VFstk8fvIRvdA1bdIiW0qvzi0yUIsjd5X6aXm7U3c4jNdmBFXKg8M0T19D6PUFyftUasdzbKyBPgH4e+T/LRE0WZzo2auKFUmHiY6fVWKPL1fL7YdDnFf8QBA7XVeyqS5nDacyVpPXx1+jYn7p0N5yiWMectYkUsRc/Mg9VaUW4jV7hwIzDjqA0eVCAWTVJ5kmWyDbi4Xx2JI67a6yafTP4yaGqu2n/+/unZ7s7PsLlqdgeJmt3oZgYfYNa41U3FuqesXPhUmOey23HA/+3exafAPwBdB64QvIfr1AuC22XwfPr8WwWs5dYzlfTZNlEP2x02i1q1Z2NYFvjV03sywp63ph0nlMYU0W93WTCEAMY5zZUVWkqeuydC0loPrOnZVA3vj85cLNZZbjmwfGx0X5G1wF8iBpk9kHMIISEwn6s15raISXKhqfXGJ8YNupTBWaK95eG0cL2bOX8JMjzMWxSUz7VVVm7W3MJsWVN9pmwZLp5RYPkoDZg5yyZKJuW6ylEItIKiAKejEpYyxZznhAiVz+xZ4Qf8hvZCHkPbtxD+4+t8vamqb80Y16WboMjzt7VOgLbLA1nXFIzscWK0uRB0/2cM4kTZrxq5jFwUtHrEyfIgOZN8eiUjhT7LH9cpog5Bh5QCen3eJR+oiskFxj3OyaJyuXPyae8v3PXbjBdUPncnFM9PkE9prT5frboPVWlOPYbWBGbhsIaEXyv0oy5bvmkjp+chreU5PITCgYloFVHKy6CIMYxZFaSku42ivDXh0ml/1bvwHt9w/fChcTc3/VxVjAyuXS2OeC1TVfyph/YpoX/CW7CC1dd4NzuadYc6GenpIWufy40n0fYm4p/VED6g9V13vx6x1PYgVrwY2P+SZ2zD+t12r6WDfuBMnGFlDsnzU0j6S1T9ki/XHwAv8wao5BcFbQU5r+FDu8Y7TYTk7mc1wT4pzy0Lch6gA+VLywgE5+qSeLtfdsn4uQFZRXZVyXDnpgPVQVcVf9OlftZ6+v0uiTrsrP2uSCKT0msLRsSUN4ApQ4cQvJHOCBa5YEmz087Jtrn1JZTVl5kPuwV2N7Z5bfd+O6xmQQnZ05mCTE4Oxfarb1b7ik6ZJBQz1sUknHHCB+2d1RGiYdJAsJHo3pyhp8lWeQcIoW8jzaVKUfLMdhl2k5Go9mlFBDvcLlYQdxl/2zSj2lhLT29ZPT3luDgr0sIcQC4aqEPuA2Mi6r9AQUttkwCQ6OUbboYyjHYt5ierPXB6+tK0393tkvDyHBVeYtncij0/dHoa0B/xR1+APa2zbNlwkFL5ogCOwCYlpJRPhFyAy4U6Kz2Cx0iUEyo3n8olaAyjMUmutbUqCcsTdcju/rS6uJRnc0RX/7XbUfsWbYgRJiIvC1pTbi9lJD4FsSvVnhvPVWxclGec8Jq8OJADO6Nc94D1/DVXQ95KTYJY0i8IYvNk43ar8WrZUJSdoOhzGqbGf7j5zaJptn//kM+W7cW0t9evt3/6cpptSFoJD4qB1l5o/Ic7nbcZpEoKtYGTWumDFLXdrPkQ0vXuhO1UwbXjUx/Q4N9/+v5vD/dP82/Pyhd2OoxbKsZgHHSKCo0+ScE5pYOnOlHNXC7xoZnkYANaFtQdwfg8Xd/MFt7l0+qbXNtgYrT3z/+iYZkarNODnK9BrDOUBmizeQXGc8YVcU4Yj/Y4TV4TSVKXsEyOYX2zynH4eNfOu+ozU9lzQw84LJE3z6j8IRcr+DoPHZDbVvXEmOT4mM0zoby6desmt6FYwuqu7UqV1bQHLmswUUZfBfZwi3PAOPDLw0xsm+r18uDjF9iMBJ+zHvDtq4D/ocWlNAp4Qc9m+yrZw/go2RON9pHZgNZ1TCEmnoRDm9pwQUBpTRwTVKXDMvQPsz4J6i4X600u9vfr7DrMcn2bKWXQ4IMO+/q4jcsv+m47YHU/X98CGmvRV+Q+FYG96fQxNfu16IeoQ4QL1gvpuBCCJCeETuCVA8uFMtq54FOQSVRohq7+wQ5VX7Da1ZZZX3ercKpi1oOPf0qpv+CTIP6160y3vlrBzG1ygur2+zldyegZhblO4uPVdn3GQLULAORTW7XmvipQp+0LwYj6drhiaxlKOaOp79nwNV1L0Wa7J1ARSimT+s7oBnYetLkFWHLrzU6XDgPL+homvxCw24Kjs8Zvj/dUnNIoK9+c0RbuTMxuP1XUTyjtQflCGD1tZpe9cLh4qVU+VpizngMg0kyne62kY3Qvp4wIix40Mzl90eVTWRFMJBpyXx5OKUTNuDlmGe1Q9es9rD5PbAsdTl1kJ+P1Lsl3u8pik3dKVhV8zHoiwoxNRwOmFB9Dg+QkaBAKrVQZBRVcUmKFSNRGbWPSnmpuvfMDNMi24gXpsDd9Hy2uthWntjqlot1A6Z7x6WjACR9FAyeR7RUYi7wfaOKgabBOOul9AM9zCU/JhIYKGuRSwjhidTkyPH3C4dn2f/v+3cPdhPhvFqsIXceqbsTwfmBpL00YqUN/S42hihGKWM9xwVoiidVRRySsiZyBooQmQlkgito3LJ26ATXBlh5vfzq8azMK794r4a312jipbfAsKaoNodSJkCxXChT6aie0Qnuz6mLz6+vcRHL3cYuyqekw8KhDxb0CVNddfANVd7wPNZZkafeI6Whz2NBiiDZUWwDKO6EkTEClQZV11BoHgZoEMYSup9BJb3mdt05Cdxs/L1ZTVUSpe8xhsxq3RvX9bR61LRmZgx8PJ7sqtstLRSKeHSc7D/LTZqoozdRnHBos+EjwUbsgEojICUpKR2dyqVmuJNgkU4jIUMAUJ8CMHo6J5Xmv12iB3blLRcMOHtFTcPYhFvZzN+hPse2iBKFb7c39vEUx3KDqrFjsrMc4fEYf4ZJMdtRy10wzoFblvjSJ20Bj7lLDZJSCSSdkjCS3EKT/3yz3Hmls6GTkkVKzUbFLzymQfCCPK0OBGu1l7iNEg3AJJAgUywAh9ZJnA3fL7D12fcXRkVy4OCVZjkxfttduHkbu2pQvVm1N0KKsaDfd7qI86NcwRAdl0TIR0iSqPNqHgjiOrjATApeONcSzCIGLw9ZOVaKmZxCydpjBQ43R9fGBKHLaDfBjE+6MyM5jWl/vQglvTYi6WW45UufV6TIUk33ZZ8RiBnDetx+yK+C6HggalclIago4e0neB+b5WBaljhJ6gkBsyYeHKcIoRtqb+b/C56vom6Vb4XUEu+asfXl2Aa9NZ0VLQ0daP1455jmPmnhr0O+nwSVHmbTCRZUM0ZpH7t56zCVCghU6Ms1mUVdgX7PC0zSETUiHcfFHLwzVkSENQEo0cIgH4qlnwZlg0A2Nijrnif6n3pl9yOPJRAp3satG0MD8w3BB/KIA5HR76LnC8yhDVAHXEh1PxpKI1hEadKISrRuuEuRogAGl9On8na6w3Hyz+byEB3FTpxSbCOuwapfTZ7kNAfaq26LjwNu5Ozcwh1UbmgfU3bk5WuKrq3/ZXv9pe3m7G/3D9t5A4+5Sq2sl3sI7Ddf0KAsUaqWmVpWH3Pl2dtNHwbnD9QpyM0XIOP6cRdZiDo88tGOeKrZRZfMdY+zrvFNarD66VXx8p+7vw0vh23we82K2r/4Tm06JKkPGbeJ5xSLqSw4cHc6Ya73SkBxnDv9KFS0QTpjlA8bMDq/diMvYM4dPOEw+y3WRv81jrnZCIrdkzErzgQSnzeKS34SQk/FbBdS7w5WxvYN5zpTb2vMVkIuy1POERpmy4/gpq3SgJDc/N4G4EEF4T4XxmhvnNeEi1wd0pMI4/ngLK7iccfw4/duRuKfA+nu+/hccd+XWdeHIMgd360lVMcb/A4n+dBI=", "reference/per_task_s3_external.jsonl": "eNrtnd1yHreVRe/nKVK+nUgC0H2A7rnKE/gVWLSlJJqRJZXEpOJK5d2HImWKDQLoPwD9t5gqm+ZHfgfdLYWr9tlnn3//9P7j17vbj7++u3n/9qf/+dNPn3+/e/f17tXbd/+8ufn+uVZK9E9//tNPX959/jT8njePn3578ePN3z59eHvz1/cf3n29/yb940v/+Pj+7tuXzMOX/vgv3TePX/jjJzp7/5+//Gbk8Stvvv3z5su7X28/fPiL/vYDr1XgG/7+/u7h1bsv/3gX/3lJ/ryM/rxW6QOo0Xcw6Xcw/js83KbUPXj2DaF78Pjyt38+/3nl//zDN/zx83+9/fA1cQBJHkBGDyAPB5DIAWT8Bqj0HVDjt0Alj6DHH4JJn8GMnyH8Dk9neP4Onz9/ifwJeHrFe/Tfvh5+5k+v+A/bLyLhIhIv8uO5DopI4kJU5EpU4lJUuIxO3DATqWMSdbyfCT6Yd7/9EnkwT694D+bb18MP5ukV/8H4RSRcROJFfjyYQRFJXIiKXIlKXIoKl9GJG2YidUyijvczsQfz39/+Vr14Ok/3+tnLgRv++Gr0QT17OfrD0UfmVZbAqV8+vHBhGSs8fI7+NauR0qGfjj3Wl7VNqrYZq21StZ9++j//9e9p4NLlABe9DlxUElwGv3Qhl92SS/xXdjVyiR8BcllMLoFf9SXIJVBmL+SiwuTynEM2QZfA7/oS6BIosyt00Ql0Sf4WT/0SH6m7Flx0AlyS1wu3PHBLI7aA4NLKkFukMLeoJLeMvMFewEUlwSUAHjvUXFLkMnKAbKJLEl1GDpGJXVSaXWJcMUSR4OPfUHcJ0kv0UpYLL2F8iRVazC8qwi8xtNi19BLkl+iVnBVg0F7mMoxKMUyyeAaIeZLy1kGMGGvyay9WdwOGcS1No9LSi04CTLo+ysshlRcdZpdIDYSXWi0jHeaWSA1kF2QXZJfpxGKbAt0i63WLxoiFbhHIArKALPSKgBagBWgZgxbXuQzQ0vrQYofQIkALnSI6RXSKjt0pqoMvKowvsSpLe0UqAjDRq1GRy0ndNROpZFKVTKSSGe1fbNsvUgmMSVa+eMMoQTIPxSeiTKf6EvpLMwtl6BghvyC/IL/QMUJ8QXxBfEkTS6vmEosZF188n+64+CKXJxa0lxNpL0YWYksd7eXHEPKFtRcj81tHUt6lGygy6tI1p5qQVq+dHMym6zlOZ9p0jWxFMNHaNRDmx3NexzBW9zkaSNZjGFF1G0jMGkExT38ttfU+Npdi1OvG+9g14uy5vRR6uOXFmuADPEdATOiGVsCh0bKF8CjwHOtqPdFLr6z9TD1HFZIKP5VNuGrSUeZSllElpqHaDpvO8SBLJSFr5ARoRUfy6agwSMWqYNTBqLPGqLOMiPDp4NOZ4dNxRkl+lnEy9Ol0CpbBqINRB6MOc1Ib88syfQd8IRJ4d44dZxqbf1zKWc+x0+MxvsAqA7ODXQZGdtDbEpYZ7HiZQaDOUce865l1TC23jpENGlAx30rxjtNlzTomi1nHNabNHwzsem+hgd69WUelAWbkLUz6LcyEBgadpFgfKPEkaCXtv5WkIiQTvRp6STvPB0aKISF4n42ktilgiulUO6uRhBjDdiYaSWxnYuSbkW9Gvhn5HmOWXpVQYFRd8wsKDANT6C9YeZFfKsgvrMZGfNkdx4gpsOygOscwkATEADHkBrNh8todpHV2GBjmoA0kcTkgpvEgputpIF3PzTsxoKWit3d3KTanay8F7zHdpnmgMxaoQvNpf82nqYErxQ3Be0mgOUNrqu9K2Gla7DREzGwi6RzbUUNXqpaiU4lzTiTonBttNpJzMNasoZeu6fsMuxTEoxd/tKktbqyRE+CLHF3MoR+1vh8l+1RsaEfVWaWwaDq7znB2W2k222KoqdSMMluOZi/Al7e3d7c3N/+6/fLl9vdXpvcD8b69/Obx1SWiizFKD7BFK83+BHQX0AUrDVYarDRYabDSrFJfBvjSqH4WvgRUl36ouvTiBvjyIMLsWnVJ7NCcCC/xdwBe5qTJyPbwEj/EiYaZauywDGlIReAlUGg3wov2PmbaY5ZstBwrWWih09iFlhFpxsqeevll+OKXNKBi55jKQJNOUkfQmXSUOYCkRWfVd+oDEq4aUmrw1BBTQ0wNTmFiak7nphniSqPW6jnDeBqjvHZU3+CiOYSeI9hott1wQCuqWivqREsOBGg5ro1GwJZl2GKdy+uiUXY4utRbVBbSaLDQYKG5moUGrQVsYa1kIW7p+yav3KI767l/R2P0hJlrhpZQW5hZ2q/YMt85IzBLLWaRFZZfoT90KGBplZPcQks3FFraK6wvQGqBWc5raakFLScLw6vQHwJayIi5IrX0bV5q0b6rRWgPwSwk24EsBw6226upBWjBiXs9ZmnEZmYWLXUtLafIhVm7nUAnmSVdn7mhQ64f0GFgidRgagiFBVgBVg4NK9b0mWGl9WClpy1EEAwWXLZa48HFz4IHFw9uPnbpMs8OWRkugdRaobTALrAL7HKC9QE1ekQqTC+xKqcM4T0ewKgEwCQrs9B6/kJrj2H6zLZcLU1d/QWDCwYXAAaLyyn4hR0C7BCAXibTS28zW13uP7wpaMPm6dJOF+Dl+BNF880uoAtrp1k7zUjRxbpGtrV5c/2Nc3pe1whmwZ2LOxd3Lpn+eFww54IqUVSRvs+NKhaDCwYX+kMYXGgQ0SCiQUSDqGiDyDqTWWzprPMIRkMwEMzEPdNJghm7CyDMzhBGRRAmejUwTE2TropATPRqjkEx2HSvyDFt5hy6rtOVOYauEU6XUzAMVhdEmMuLMJdyu8Aua9ilb7qVq4o6j1168djFsBqa1dBXghdt9r0cOrAU6Xr48viQTrIe+vFiijtggoWKeGCClSpsLnqqWtkHE69cwQmTuOwKXph49TlumL43Kzmm8TnGVu8lreSYxNLHiRwTf4ej7FzcHmMSt2AyxjTex+ZW3slH2sVCxjqAE7qUHQFO4IEV38+4oNskOGYO12taRzqBP5gbYc+kk9RhIOt9rAUiZzJn9zZKfGFHMNdgrsFcg7kGc83xE/BI8KU5ta8JJ2eaJjPCtBaEuWIAHgPZJ13wyEg2I9lAC0PZe0GWps+NLNZHFsvKJGSXI8kuKs0tI6dAd9lEd1EReolVQnih0cRQ06GNwR/ef7x79fbdP29uvn/eingD2n98y5vHTwNE074kGhkQjbbG7yN1VyAalSaasV/nIA1IA9KANCce1A54gWCarEyjUkyTvuoMUKNSUJOsnpdqrGrnUo0Z02nuqaavTjVnsAuDNFkMwxmIRvDGHMsWHChUyBpzGiNw6FKK6DOXYRnZyBoTHfOqIc/EileWZ6ydLc8EGk7KAxnXVAYZdkTef8vPtz9va5GJH6De/HbiJtTz+MYPUYJjHqsVdskEihQZ3w5dTBGHb6DQKMbMN8o8Vik8vB0oUqTNFLqYIm2mQKHyCcBPRWt7ZWKFazSZohddo8kUKz6bYnqnMlCML8c4NaQY07J2iQA9AvQI0DtwgN7Fl0WyxIAZpT36ZGzT2ew+GaNdN0CYRtyYEGNkFcOo1251S8nJ9lKMOXwGjVk3rfT8MSzGGLOD0Bm7eFzJnChmxi1pKJkqA0uPj2hmroyZ308yMyeWvt+0eRRj5o4shapMoBhTKULPHGxmaV03yWyXI9NuGRyzYGbpJcGI6TJ4YsyQYIwdijCtkvN7Ypi2fsk/9aet40eoiS9ysmnrH8BTdNo6UKYQvOw0AnjPqXi18n/lUvCyYfbvlsm/kgVeetVk7yCZVrUDeHFO6CCV7iDpJLqk65MTs4OcmPndIx0Gl0gNUmJIiSElhpSYo6XEvIAWp7r8thcj/VBx6VrNbDWz1cxWM1vNbHVRDwwoQ1TMZWJ6X9KMcS4/zVjTeDTDKBIaDBrMTpZIVkEYdBh0GHQYdJhS5NJ1/WznSzNKLr3XPNJKlY+DiW3oW+qEmfh+08NiAjuuyI45h2ST3g5JlswBBJzIvsjS2TLhHXxlo2bGaubHpQnXWYKexssWhKnUdsV6k1ETz1EjsWbaDalAXhOPUg7Ebr/effn0+febm++fvNL3yDRQkL6/8Ob7v5foR13TDSms7RkCZwicIXCGwBkCZwi8FPgwA35yFeklvDRquKkyC7z0ZgAvohRWHqw8IAw60Bn2bdMI23Ej7JAIQx9sDcGYxuYmmLb1CcZg38G+g33nqMPfl3fv7NV8XEt7wcADuOwNXJqhf2ccXALuHeuDS+eBS4N553zmnYmelVpdp8kHqroVKruxJ/bn6Mom53HzTakNUoGnMTsAMPwWM6BqCwtQ4NTziCty6BmOoLEjTGiFRd5ijkFo7BQTonq64Z1oDzc5ltEhpMP3cwnaxZ7NwimzSQeb6h+adLLJEYZTTjYTC1tp8utZrjoW0pHbhacIUeukohbJQDTiaMShZ+1HzxLXZwcXEQ9cWsAFKxFWoitaifBDEwhESw47dEmG6V32Wa62azyGIY75AqskEF6qBzJjJrq2mQhwWS296AS3pOqivGxILa3q81ugOzOPWrBAn2EEHWqBWmgX0S6CWWCWssyiO7OaWdqVSgvdIrpFWbtFZg+7O41UZBcjxXdghYqUWd5pam3vNFKeXp5tKy+HL6EiJfgleDGFdo/LBoYXI5sgTLRwlf2dZssFnkZyUEyjsnteRNshxWhFv4h+EcoL0+c0jNKjUUF6iVVZOn6uIvgSvZrF2X8qwi+xSiZSyYzuY8Lvcon9VyGAsbIaYIwHMNICMBcML5at+SV+BDpHK/BFmDPaZ2yx0Dli0OiCraPGzhRdzDiz6G4es+yidSSrs3Hk+GE4snn6jewh7kZW941k54E2UkF4CVSZ0Dea3zYK1JnQNpL5wovUiKSRCql/oUspsnEhUGhC1+hwosvTZW6wTkpWiS6yKv9FVnWNsjSNRHXZm0ai0VxYGMWINAujGJBmYVQtfGFj1JFaRiolvzwUn8gvTX7Tiyj4BX65Kr8wb0TbCHgBXoCX0r0j6bv8fhc/XbeHXc5v2N0HvOxh4EhO59itNG9Ua9xIsLzMQ5das0by5wvt6d5w0kiObnnpe8ktuZh+aNNtuw5sIeCFMSMEFwJeCHghTRef7ipocdrmTqWz1g2llqa3MAvMArPkZhYdZpasuS5sMGIBAJNFEMt+iKVpshOLMz6x9LsfLIJZYBZ0FqAFXwvUArXsnlqsspktLbbTw3GidmxhNDoLES4wCxEuJ2KWGuG5JLgcOHhOQJbFyOJ0dqHF2cpCC8yS/H09Wh+ZhdYQKgtmFogFkWXvxNI55TJnzjndDtcVtU1XvDUk64hlfeTcCQaHiJy7yOTQuSLnKo0OLYmcmyu1LEmcmy21LAuc2/fk0FHj5i45OpQlba7rnMncHXLa9UNwaTVSS4X2kGwqtRDwf9oFRacad97tfiIaRJtYWrbZrIjYMplZvn7++/uP/3r19tOvNzffP9eqbQYDzz++583jp0t6RKLMUHHRncbWwmpFmkQ7kVuC5BKrwtQzM0SnRhed0FtAlx2jy3DuORe6tPPQBc1lJ82iFLuMnICMXDL+j5TxzzQRKbmk5B4u4j9IMVa32Smm7bzOkXbCNDQYM9H0ksSYsbtwgqz/U8kwKgIy0as5H8ng2MWxixJTUImxrpnLMM1LhhFfifEZpnyiS+N9rHXxWu9j9R7piW84mXdiF7zQOWNWOWfil7fQSNOua0dNPs90fcfuYI2AXYxFgT8t88QeM3OiKfgE5kk/Zq7nZrzmBCHIVrIN231acIL3sPzm6rGie5KJzEK+Cvwd3EA0it3rRRpSu53ruN3QdWxzwJeTXmVgLzvUj1yjvS6YG2Ovieh0JT9PZrhiD9PhG2QZEIp+WZ2d2LHftFftn+kIXJUewBqteyhuWtds0ynoqgtO045SB6SmnWUuWLm2y9CYEw+sHpY/PQcrlkGxwxJ2msROex3rgpYW2YtMheC/lqbcYa1FZju6sUfvyTknLr+taDa97MBWpNL8MvIW+IrwFR0NYTAWLZZ86vTGghgTq7K8GxYGmejVLO5/qQjKxCqt80lvGa0Tp5lkZZzSGZzSrteqgFFa95WJhogd0owZVGdrFO5ohBim1M/ujXZ9JwWopbFQyxX3Rm3fRWJxFC0k8nUgF7Zdnh1dOm1mCy6jyxjazrVz0UUwwJzAAMPKy5WKi1x45eWuM41lhysvwZZQKjGJxuenFmP7Enk6DYILicZgy0kEF/pEyC00iuCWHXFLKyW4pZfDGXbxt4At+FvYwwC4AC6Ay/7BRebPSY+3ifrGB5eeNtG43iJHN7jQJaJLdEZqWdIlYv1CxfUL6zpFl1y/IIdHF2t0fs2lb+1cdGFIGtEFeGGoCM0FzQXNBc1lDFx60xSYhe79pVFdewVwUWlwGRurId5lT/EuKs0vI6cg32WTfBcVoZhYJQJeCHg5WcCLSiFN+qozJLyoFNQkq2eNeOmNdGfAGry7MA2RdUTWEVm3V6Kp3Foise6KiXW96Uq0lzp1OJUGnoFn4Bl45mI8c1anDPpMZX1mJzzTmBJdp66ZxzNsRMIrw1z17LlqHaaYxPJtnDLE72KUwShzZKNMa3WG7djOQ5bezJVg2I69Qz1m4n7uI6141K3pxD375+aMM/1M+/AHj6/o3k2Yb+jGlqeiCXXHp6Hq79ZWryOLpKfPSoWuuzhUqddeUVdp8fZY2YkBfNuJRamt0Acas9pUOQr/IVgJZdKX6Iv1Fvcy7mXcy7TGTupe3ukUFp2xg+/ZxuYzoy326/v/e3/36sO72y8f74Hm2X+90qpr2wHXPHv1zfP/WMI3je2GiTj3wIONmQhiWmVk+R04FQf3MvNYp8OawzXOUlDTq25HUIP55wR7LEEa1lji/0GjQaNBo6lLM1pcQZoRb7elKIFmoBlohh1RwAxuZtzMuJnzYIyRTophTKukG2JMa+g0MZIFxjCSxUgWEAPEADE5IKbRJqddxvgQ03gQw1h5BS1GNmWYHeyPStyCk0oxUpxhWB+1RImR4gwTqFFmAkpOyzDRLUqFGSZWF4aZyTBNVSEGhqGfhBBDP4l+Ev2kfWHMRs4YtJhMHNOakhzj5nEMo9mMZhNazGT2QUKL6TDtGGuOOb4E1+ThmsaU8/u2Rg9XMYho9Bn0GfQZ9Bn0GUAGfQaOycUxbW/LeWWM6uZxDIbfU0TL7COyeA8RxXuPJK4izCwLIa6mywimGWjmqqaZ6CUfEGa+eXMLRso03hC2lis0m1iQuQ+eISsPnmGQCZ5hkOky4kyrOleSZ5p5PEOTiSYTJEOTiSYTGEOTCY6ZzjH3/yvJMa6yLgPIkCpDqgypMogxUAxizFUgJq9TpvUgxnpOGYPjt4IYY4iVObRNZokYY06TKxOKyCmzmFtOs66gUrCMO3OyjLmgS8adyCZjTV/M89vYzpNjDG2lS1h+QZnqKAPJQDLniMgL1DlvW+midt9Y7cUc4+5PXa6t1IvHMRaOOb09BqcvKyTx+eKQwSFDc6mmJNN3BSeXnPHiZAzLsEGZOkPYx87HA2bOFI7HRmx4hoXYVYjGiFaLg3/NS6JphsEyrtEDorG6J/iX4N9DBf+qNNiMnILk302Sf1UEbmKVFtONCtPN8x/JgDcqjDexKkv55kd3pzDfBApNaD2F+Sb9637TAGCVIJxk5fWI4zVx6iJOrPjU/lMCcR6Kz2Uc15abz75nHLki4+CloQlFE+q04011uAbZ5ohYQyNqe9mmd025PGDX2MpIgzkYoGFVE72owr0oqKYa1ciFljVdthuV2Srcm64r6BTumnkTT1DNKaiGCJpDhAITQnMelQarMDE05xdofru9+/zh092H97/c3Pz4/GFweyDO/HjtzY9PAwjTvEQYO0AY1/VDP41ry/eaGu9jHdGo19b72B5wYle4jZ948nlqaTrTb1A9iSf+x2ix4qPT7ziDlgK3q3CLa7xmAXVowoUWEYuCz36udqQj77Ez9AreY/ply/pl4VtZ2/Y88Rzldafg36ODylA6dSlrYM6odqhJTYW5UT3KWWOGMOdaxr1osuEaOqoehW3oXHZofEO02A458BVlGetsIZbRQ8eQe8hJxgTN9DqWISxDjK/jGCrGM8yvH0uqWTDcFeMZLbaQNtOZ4VCX69BmgBlghigeWAaWgWVgmdws05i+L8MyrvNYxml8zwdoNJHAcx1x5nQBPEx0wTQXjt+Bar5TTatNKarRlalmfccJjQaNBo0GjQaNBo0GjeZ4NNMqU8g/0ynfCywEJR8jKJnxdOzAjKcznn688XQSBK85oG5a3ZbqNfXO4xhDr6m4KKOTDJOuD8KAMCAMOcjoMegxBxtnasWpQhjTtJ7912D/ZTSb3hK9pWv1lnSYZSLAhBrDHs56HHMaNcY2LhPGGB9jOg9jGtSYChYZ2VSNidevpsYkbkE1NSZ+hhJbqSSEMJEaSwlGamgxoUspocUE6hxXi5EaABMocyKAkU3SjmN1KzSTopdcAV+W7G2I4ku3TIUJbA9vPXyRft4Q9rP/y1+kwsR/fnK6sWyvwsjW+cWCCLP50JLUCB0WRJglE0tS3uAbKDIKMeZUFLN8A5UgwxzI3msykkxfrJ/U+raYjn4SrhhcMUfcDo4pBlNMJVMMi6fw9c4AGNGm1LS19aetLQADwAAwmQFGhwEmayMJftnpjDVGGODlwvBiulw2GL+P5Dx4sT3wUt4FY7DBXM4GY/DBXNkHU0t+MfvuIa1zwhisMAe1wkjr+kxWGM/Ja63fQFLF82GEiSQmkkjwxQ2zNzeMFB9JOpmj92JeGDhmLcc4J0s4pnnJMZ3PMW11jtlFzl1g5zvLCdYuJ4jd1MWgo81awUa9DmyiXwc+8UPtI1am2u6CwOOuQELBJzoXjLTZqcoTvKsXD6JZCkvhO7mInfRmIpDZkJ4atyU/xaovIqjeFrIS217PS9ijmYUTh4A9Avbw4uDFwYuDF2cKvljT60JGYm0rBwSfYm0TnSwYhnEoxqEYh7pMtB4Ysx5jmjbXPFTjJ+s17DmAYvDj7NaPQ0Qw6yeXoEyg2VfDoDNW9sQLndhNmWM3pW1trt2U4ofX9JU3bdNyAnVYhgDpQDqn8NMY9iHAMiGWefu/tx//9unm5vHfr7RS/WAE/PHrbx7/taTd1PVquAGhM/YKfmMMMxhmiK6h14RfBr8MjaZ8jaYXwNLrdiWwqAGw9Mo5D1gU9t7ifaWfb3/eeBwqfoLp41BJYhm7Cyp9G6bKLSrNLCOnMOlTlGgtPVYrPfAUqDJh4ClMLtGrUZHLmTDiFGaXWCUTqTRh2ns2vTyWmTnTFKSXWBUJVxlVXVSEX6JXoyKXMyq7qAjBxCqZSKVpC7VXdZGeqi5RXlSCYpKVJVF5mvSiUhyTvmqVuuxp2otKkUyyuklVnyW+aN32K1lm2DjqtbKwzAa5exvH7lmUl9o2X1Mhda9jUGl2v8hUSKrp0F1WJ+6huxxLd9Gm6bM2iu5ZpZvHKswlncPRS6eI0WqIhVYRQ0kwS1lm6TqZwyyBUOB+yCymVR6zSNn92OfoFa3dj629j1Vyy/pt2ekQvLk0k7g9kz27E080PVJPVjt4Jx5pnHXm79UO/HGZI9UsyRWOBOVN5qDQZY57fceqjmJRoOwE5+9Y2XFHzfwE4sAznSPzLNnOHYnpm4xQocsc9wi33kelaaixshXSi8OPeIFEFDvGNDtxKlNwHn1Fb8dEc3H4mWyTdDz8sOvZrBedV08yrfbYrKX3dfqZKcQkMm7wHZ9xXAo1CTVpT2qSsbbJSyxN67t12JLJlDdT3kx5H3u/FNl8lca81QZ2Ywa9Dzno3agmM720jTc31RLIRyDfhfjlfAae0/HLbmWXBQNTZAsz8X1F3aVplc1MLp0ZkotodBd0F3QXgoSJ10N3Kaa7kK93PdnFdZnhRfy4GsHmgs0Fm8sxbS61wIWZKWamcLmgtowCS6tkbZ/I+MDiuVykAVjKA4uQBnw1XhHSgHdJK1I8kwZYyQArQmfoYKzSNm1eccU2ap64gqcFTwu9IRQWPLlwCyIL4DIVXPrMZlzbGrpCROkhsxCld1mhRYeJBaElP7Acc4IIYllMLCLO5CWWXvcesfTE/x6BWVYvXkJsOQi2LNi6xAQR5MK6SLhlN9xizay1Be1Lbmk9bnmwr8ziFrMuAzieITxRakmEEE/EFvXayfG5ZW3ur5Htk36jZ5ie7evWh/tGT7EXcDHzI3yNzA/tdUtSe18WmpDT65YE9b6sdKnp52UBvNWmiIws1F7MmpRdIxvl6kYKT0zSdeuidGNXPQljotWnDhJFqs8CmU7n9bpopVRbWYGhaXQSASbD5mtGoXckwSxafI3jhWloUugYh54xDi1d5tT/bxTj6TF2/2skVRpjRt7CpN/CTPj1Cchg24VjCHVhCcAljTCXhZgcLSVrxK7cKtn6DOMvArCatZLl10puPnQkmHfXuWDm74Y05YeknzVripp3Q3X2s7xIyKLbdxdJyKKbxy2y4f5Ft76DZDulsya7aKXFmzqywtQRWXRwCysXCXfBucuQNNbdFbzSq9y9Iu3UPF6hV0SvaGemF5UGl5FT0C3apFukIvwSq7Rz20s1/26YYqJXg/Hl64iBN04yyco0jdIw8//O/IF0", "reference/summary_s3_external.json": "eNqtlM1uwjAQhO88BeJaVGXX++PtqW+CSkVVJECopSfEuzfgkCbGaylSj/E4Ge988Zxn8/livUdefWx3m+/Fy/zcrrRr18fV5/b0Cu1a8xzMlkk49DuBwzLf3Nx2x6C5gkmxBnOFb4KCDYWvzfvbbnc3Zy1p6YtqUhA7u6glMTmKUCtdln0CP4ftyU+AVKckMDYeJkDRSUDBTYCC+AnEUEnA0A1A8T7RdfLBqDCAfQ8FGB42JwfmXOishTBXkjGN3hgNCmwlLX2QBAti58ZEBTEZBtUe9Wa/frridkFLoAmgTckD7XDOrsBofI7gczbSCmfxQcdHRN3FangCaInmkI4NlElLE1zUiOSjZjUftZiPmiINUfuUacp1NoweZYEy5ih+of1NV7rO4x8nw0zgYwYrY0bFKZjJw6yBy5hZKpgbrmAenyLDHNDHzNhjPh6//qu1JUxubeFKa4cKZuRaa1dqG8SpbZEptY3B7W0rYw4GfnGPT5UVd59eqbiL/0DneKM8u/wCKLsqYQ==", "reference/diag_s3_external.json": "eNrtWNlu2zAQfPdXGHqOAy6XZ3+lKAjFkhM2iuxKcpsgyL9XPqqbtHy2BfIUamZFhuLs7NLvk2kQvxZxloZJ8GX6PplOg6e3h8xG+6dpkJYj4HC3fVjYLC/MfLleJXFhfmXL9NE8LpNoYZO4jKN8F9YmzCIJC7NObQFljHbO1Jtg6L0GY6O8xL9u4GkQfQ/Tx6Uxu78zAIo6uBvmkCC6OMaQuTguaDXnS1iskmWR2Adj6nH5vqB4IIYSptShGAbs0FpUIKN/YlZvUViExryGWRa+zZBo7qKQOCkhpYOSqLGmEpsWsyj+acx+zDgHHy0Iq+kizit6OwaCXHh4qVW10Xxun20xS+IwS41pPpUHC2xUHGscpCeOciD1fKsnm77OouW8jNqNgQhgHl5KLn20BuKhNUU/zZWXVhBs2G+7vIns/GmXS8amVb6RfTJu0mke2/KsHs2LzXO7ZZE03t1mcoNzUO2sPKCUU3bd3tb2P1/Z+XNOOtvZgZV9NLdSxdMeWMVDa38br1qnqzDL46gzWZvsrtf8shv0XhFoMFX0fWkbJfyx4YKHF8pP8l8Uh/1X+Py3PcHQe2P9lwqB1/TfHqfA/Z6qJdflhCZwK09HqvUlfJ+XxvIv1QZGNHNQQlJwVRRan+ZFi40kinhopbQ6pxYhZZ+l6uKlykNzfXYl48xdyUCNKmXHGM4IU2mZwX+fNM2kuIDqr6zmm8q13aHAkR0KnNqi0NNbFMHoYIvCQW5alMm2SwmKLLSp785IR94Z1eGeBYi3aSHOpgXA37UswrwIV9YAQF3CKpDqWq0ViJqSHsjKdqcPMhyIZLW4GqCSfZADHQAV9sFGf1ODmvcjOaEDIFe6D0rFu6DmvI/Jepks/rEu3SA3QjDWBxtJWYESgfdBXnch2UbA5UFgC0BCRAcQ3Qgh2wCrDW4HcGyvUvpnF2hsYwfw9iqMSHHNW9a40uRX8bA2Gzoa+wGP/hrnXc7Q7XzE53xwuvNJ7nA+qT2Xs5FGh+zcy5l0+pwcaXPnOhrKAUtioM/zPnpxQ5RKfLrkSS4JXZdkQnUBfcBYNZJDxorqLzgtU26npeQ4qx3MJEd+DKbCjVQ/WsstiV5PLH1tnFdV1JFVhd/+Jz9BcLCqMNy305OPyW/pTz+7", "reference/graphkey_bm25_ceilings.json": "eNqt0E0KwkAMBeCrlK5Fkkz+xssUrSJFHTftqvTuHRB0Ntm5TcLHy1v7y4tkeAj0p27tx3O5DkuZ5mF8L2WuM4R86Nr5bXpO5V43cDRCF07AmsQUdauXH44iLluoKVNyRpCsQMq50SgMhzGn7sb4zdhw8lcuxdVJzGVJzcM/jWMtrs7AFTIZCiU0923bAe52i5g="}')
for rel, b64 in FILES.items():
    p = WORK / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(zlib.decompress(base64.b64decode(b64)))
os.chdir(WORK)
sys.path[:0] = [str(WORK / "src"), str(WORK)]
print("working directory:", WORK, "|", len(FILES), "embedded files")
print(subprocess.run(["git", "--version"], capture_output=True, text=True).stdout.strip(), "| python", sys.version.split()[0])

import g4a_repro as R
R.ensure_deps()
R.restrict_repos(REPOS)

## 1. Dataset and task list
Downloads the SWE-bench Verified test split (parquet) and checks that the seeded sampler gives the same 153 task ids as the project's saved list.

In [ ]:
R.download_dataset()
if REPOS is None:
    R.verify_selection()
TASKS = R.tasks_in_scope()
repos = sorted({t["repo"] for t in TASKS})
print(len(TASKS), "tasks in scope;", len(repos), "repositories")

## 2. Repositories and snapshots
Partial clones (`--filter=blob:none --bare`) of the GitHub repositories, then `git archive` of the `*.py` files at each task's base commit, exactly as `g4a.external.ensure_snapshot` does.

In [ ]:
R.clone_repos(repos)
R.build_snapshots(TASKS, WORKERS)

## 3. Ground truth by symbol kind
Gold patch -> enclosing functions and classes in the pre-fix source (same code as the S1 ground truth, no provided graph). The paper's table: 263 sync functions or methods, 30 classes, 0 async, 44 module-level edits, 1 new file (338 symbols).

In [ ]:
R.run_truth(TASKS)

## 4. Baselines without a model
BM25 over files, BM25 over units and personalized PageRank (BM25 seeds, our call graph). Hit rate = share of tasks with a gold item in the top k. The query is the problem statement only. Per-task values are compared with `results/s3_external/per_task.jsonl`.

In [ ]:
rows = R.run_baselines(["bm25_files", "bm25_units", "ppr"])
if REPOS is None:
    R.compare_summary(Path("results/s3_external/summary.json"), ["bm25_files", "bm25_units", "ppr"])

## 5. BM25 candidate ceilings of the identification keys
The flat key shows 20 cards and the dichotomous key 10 files. If the gold item is not among them, no model can pick it. Counts are over the tasks with a gold unit (151 of 153) and are compared with `results/s3_external/diag.json` and `results/s3_graphkey/external`.

In [ ]:
ceil = R.ceilings(TASKS)

## 6. Repaired graph builder (optional)
Builds the stdlib-only AST graph at each task commit and reports size, edge types and the share of gold symbols that exist as nodes (async and module-level code are the cases the provided graphs miss; none of the 153 gold sets contains async code). There is no stored reference for these numbers; they are printed for inspection.

In [ ]:
if RUN_GRAPH:
    gstats = R.graph_stats(TASKS)
else:
    print("RUN_GRAPH is False, skipped")

## 7. Embedding and RRF rows (optional, off by default)
Qwen3-Embedding-0.6B must embed every function and class of 153 repository snapshots, several hundred thousand texts. On the project's Apple GPU this took tens of minutes; on the CPU of a Kaggle notebook it is far too slow, so `RUN_EMBEDDING` is `False`. Set it to `True` with a GPU session if you want the embedding and RRF rows; scores differ slightly by device and precision, so this cell reports the maximum difference to the stored table instead of an exact-match check.

In [ ]:
if RUN_EMBEDDING:
    R.run_embedding()
else:
    print("RUN_EMBEDDING is False, skipped")

## 8. Verdict

In [ ]:
ok = R.final_report()